# 38 — S6 complementarity diagnostic (not a submission)

Protect notebook33, user-confirmed Public 0.958. Evaluate independently retrained Nartaa S6, using its exact public component-B worker and checkpoint; no new model training, changed pooling, Raptor fallback, rank fusion or submission file. Original publisher reports S6 Public 0.950; this notebook has no score.

Attach exactly **three inputs**:
1. RSNA Knee competition, including `train_series` DICOMs.
2. [nartaa/rsna-knee-s6fall-weights, Version5](https://www.kaggle.com/datasets/nartaa/rsna-knee-s6fall-weights/versions/5).
3. [heliosli/rsna-orthofoundation-fold0-epoch7-20261004, Version1](https://www.kaggle.com/datasets/heliosli/rsna-orthofoundation-fold0-epoch7-20261004/versions/1), supplying anatomy/code.

No R6 bundle, Raptor weights, study cache or prior Output needed. Reference predictions are embedded from results26/30. **GPU T4x2, Internet Off; Save & Run All.** Metadata pins versions but may not replace inputs in an existing draft: check actual attached inputs. Allow up to a two-hour diagnostic budget, not an ETA. Strict stack probing has an additional subprocess timeout; no guarantee of GPU memory or wall time is made without Kaggle execution.

Gold58 is reused, not independent CV. The stored ensemble reference is **29B proxy, NOT anchor33** (Raptor224 gold unavailable). All comparisons are descriptive. Visible predictions have no pinned S6 reference to assert published parity. Failure is recorded, not replaced by Raptor. Send Output ZIP, especially `38_diagnostic_report.json`, `38_per_target_metrics.csv`, `38_s6_gold_probabilities.csv`, and worker logs.

Credits: Nartaa S6 release and Heliosli code/anatomy; inherited publisher/third-party licence conditions apply. S6 learned weights are declared CC BY-NC4.0, code Apache2.0; these declarations are not a competition-eligibility determination.


In [ ]:

import hashlib, importlib.util, json, os, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
OUT = Path('/kaggle/working/rsna_38_s6')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
REPORT = {'experiment': '38_S6_complementarity_diagnostic', 'status': 'STARTED',
          'training_started': False, 'submission_created': False,
          'full_anchor33_gold_available': False, 'fallback_to_raptor_enabled': False,
          'public_reference_parity': 'UNVERIFIED: no pinned S6 visible reference available',
          'gold_caveat': 'Reused diagnostic set, not independent validation; proxy29B is NOT notebook33.'}
def save_report():
    REPORT['elapsed_seconds'] = round(time.time()-T0, 2)
    (OUT/'38_diagnostic_report.json').write_text(json.dumps(REPORT, indent=2, allow_nan=False))
save_report()

REF = {'uids': ['1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149', '1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269', '1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052', '1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580', '1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238', '1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935', '1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840', '1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756', '1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604', '1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581', '1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945', '1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819', '1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973', '1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492', '1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803', '1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891', '1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903', '1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309', '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192', '1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711', '1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378', '1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404', '1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645', '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465', '1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924', '1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608', '1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550', '1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911', '1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945', '1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742', '1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683', '1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500', '1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125', '1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187', '1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274', '1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132', '1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450', '1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363', '1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710', '1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395', '1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236', '1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910', '1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039', '1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327', '1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774', '1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002', '1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921', '1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220', '1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995', '1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190', '1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003', '1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828', '1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254', '1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037', '1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435', '1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767', '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312', '1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662'], 'labels': ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'], 'truth': [[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0], [0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0], [1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1], [0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1], [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1], [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1], [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1], [1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0], [0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], [1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1], [1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1], [1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1], [1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1], [1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0], [0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0], [0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1], [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0], [1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0], [0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0], [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0], [1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0], [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1]], 'r6': [[2.53914573e-06, 5.55382103e-05, 8.59971806e-06, 0.00113390537, 0.00089232513, 9.20057282e-05, 0.000136878982, 0.66231592, 0.68918848, 0.0098157946, 2.10974963e-06, 2.01444891e-06], [3.8764585e-05, 0.00111648801, 0.000732976419, 0.999208152, 0.0046745264, 0.879773617, 0.999993205, 0.836062897, 1.0, 0.00865006661, 0.999747455, 0.245500997], [0.0250542816, 0.00265271612, 0.999999642, 0.775026023, 0.999999762, 0.999992609, 0.999993086, 0.102916289, 0.94995296, 0.030769139, 3.76081043e-05, 0.00021062723], [0.000677336182, 0.00011701147, 3.01035789e-06, 2.91227952e-05, 0.00043118879, 0.00208276068, 0.000990318717, 0.199304844, 0.809358537, 0.0126847392, 2.18187029e-06, 6.64567381e-07], [0.999999762, 0.174511954, 0.999998569, 0.91655153, 4.15807554e-05, 0.00016519046, 2.21754249e-06, 0.00210258238, 0.00252679409, 0.00673975515, 0.00285861967, 1.85706886e-05], [0.0273092128, 0.712043345, 0.0013785332, 0.930462182, 0.00013183376, 0.000151160377, 0.000336554367, 0.707510288, 0.999999881, 0.00825018139, 0.999997973, 0.918663263], [0.999998569, 0.000219583526, 0.000206183468, 0.0247125197, 0.00240426254, 0.991974652, 2.3907749e-05, 0.973201586, 0.999987006, 0.00735876007, 0.0867723972, 0.000626800524], [0.032333225, 0.999955416, 7.03475234e-05, 1.07172882e-05, 0.00163350417, 0.00482789986, 0.0123654511, 0.141171609, 0.999876142, 0.0274987922, 0.999999166, 0.999993205], [8.70375516e-05, 0.00257645594, 0.944272578, 0.999682188, 0.999993324, 0.999890685, 0.999993205, 0.547965016, 0.0236632321, 0.211045617, 0.999453247, 0.999997973], [1.35622486e-06, 9.35720964e-05, 0.000204198092, 0.999966979, 0.776146948, 0.999999762, 0.999999762, 0.951694704, 0.999996305, 0.996313949, 3.08816707e-05, 7.11845714e-05], [3.57927966e-05, 7.49834653e-05, 0.000595688587, 3.99823584e-05, 7.09258748e-05, 8.13922452e-05, 0.938468099, 0.082391355, 0.0117911762, 0.0163314694, 0.0177294556, 9.66540247e-05], [0.999975204, 0.999950767, 0.999978304, 0.446785927, 7.04706617e-05, 0.000837415399, 0.00129363593, 0.754259218, 0.999019146, 0.915177816, 0.999870539, 0.999929428], [0.999996543, 0.983425081, 0.00173894037, 0.999977827, 3.6134039e-05, 0.000759164395, 7.47471568e-05, 0.0185223697, 0.999909997, 0.00723882367, 0.00371206389, 1.74409415e-05], [8.09088124e-06, 3.22675733e-05, 1.0, 0.0151260532, 0.997158766, 7.92612263e-05, 5.93582126e-06, 0.911921936, 0.999976754, 0.988001484, 0.986728847, 0.999976158], [0.000102624232, 1.81916457e-05, 4.53515277e-05, 0.00066238927, 4.76714831e-06, 0.000882061548, 1.83023155e-06, 0.00394672659, 3.50408627e-05, 0.00906039927, 0.689235508, 8.2582992e-06], [8.62716333e-05, 4.92731815e-05, 1.05988765e-05, 0.979548812, 2.02953288e-05, 0.0479652546, 0.000752991822, 0.200387452, 0.976792395, 0.0397170931, 0.999805152, 0.99999547], [0.980021894, 7.34583373e-05, 0.999478996, 0.999999881, 0.999999881, 0.999999762, 0.999998212, 0.921891634, 0.999996424, 0.990330004, 2.9064031e-06, 0.00159848924], [4.65749099e-06, 4.07003927e-06, 0.999998927, 0.999999881, 0.999983311, 0.99999702, 0.999999523, 0.525198056, 0.999919891, 0.0149800754, 8.07717413e-07, 1.01927662e-05], [0.00359209697, 0.00171308091, 0.999999642, 0.999999762, 0.999999762, 0.999981403, 0.999999642, 0.898047421, 0.999998808, 0.874477208, 0.0297480412, 0.999961257], [0.999998808, 0.0225905795, 2.314837e-05, 7.69336402e-05, 4.31393073e-06, 0.000114137511, 8.20060052e-07, 0.961908765, 0.999288023, 0.0125384454, 0.999999046, 0.999872565], [0.999999404, 0.99928695, 0.990218937, 0.0474464484, 8.20611422e-06, 0.00842486788, 7.09206188e-06, 0.605584038, 0.561016083, 0.00779654813, 0.999999642, 0.999865174], [0.0024684032, 9.6286145e-05, 0.000117558135, 0.999999404, 0.277922094, 0.999405384, 0.0194388349, 0.0907667887, 0.000813107414, 0.997368449, 0.886035144, 2.02063384e-05], [3.77505808e-06, 0.000214428641, 0.999996424, 0.9999578, 0.666070282, 0.00376552832, 0.167065561, 0.901186716, 0.999934673, 0.160647203, 0.000243753806, 0.000580351916], [0.999998212, 0.000340836385, 5.29407189e-05, 2.88414885e-05, 1.68874226e-06, 3.34813626e-06, 3.68670499e-07, 0.272433421, 0.183858782, 0.0112239159, 0.999995708, 0.00521993497], [1.1717606e-05, 1.36160525e-05, 0.999999166, 3.76370917e-05, 0.236537412, 0.0609180741, 0.157473683, 0.00050966817, 3.98391348e-05, 0.0103875548, 4.30425243e-05, 1.28361389e-05], [0.00026542382, 0.656391561, 0.999999642, 0.0410939679, 0.999999881, 0.999945045, 0.999979496, 0.500628509, 0.992124736, 0.994547623, 0.00228724186, 3.88180779e-05], [0.99998796, 0.000234793755, 0.0112052066, 4.51732603e-06, 2.84848738e-06, 2.89649438e-06, 7.8411216e-07, 0.018836379, 3.72790964e-05, 0.00983720155, 0.995985687, 0.045192983], [0.999999046, 0.0871872157, 0.999997616, 0.999993682, 6.22082662e-05, 0.015629366, 3.46778106e-05, 0.96778442, 0.999993324, 0.0074688013, 0.999999881, 0.999989033], [0.99998033, 0.998630404, 0.00626319461, 7.01395329e-05, 6.00408475e-06, 4.0617183e-06, 0.0194192752, 0.981442554, 0.999997735, 0.0282868782, 0.999998689, 0.999999642], [0.999997377, 0.993678033, 0.999999881, 0.999853492, 1.07044179e-05, 4.0022449e-05, 0.000197404559, 0.950482461, 0.99999249, 0.0111751269, 0.999999881, 0.999998331], [0.000306905393, 0.00979625992, 0.000116100535, 0.000337060745, 0.000221077804, 0.0494477898, 0.000539784087, 0.0293928076, 4.3577169e-05, 0.286741581, 0.999999881, 0.99942708], [9.01445128e-06, 1.05417903e-05, 1.09507419e-05, 1.0, 0.0247242898, 0.999999762, 0.999940515, 0.97100783, 0.999999166, 0.0123596105, 0.00251364335, 0.468003184], [0.999998689, 0.999822915, 0.999999881, 0.0752458349, 0.0933510587, 0.0201860778, 1.10403334e-05, 0.0748529032, 0.999792039, 0.00856043424, 0.999997377, 0.925361693], [0.000276213745, 0.379888505, 0.999999404, 1.94007976e-06, 0.978698969, 0.0112949833, 0.235309482, 0.842198543, 0.999965191, 0.045819036, 0.999155164, 0.999957681], [1.67879932e-06, 0.987989545, 0.0779701993, 7.22460027e-05, 0.0474649891, 0.00303698168, 0.00511245802, 0.154833873, 0.0141653996, 0.0187076329, 4.20706465e-06, 3.26793611e-06], [0.99999845, 0.948116422, 0.0419190675, 1.38388828e-06, 0.00523619587, 0.000895284582, 4.81250318e-05, 0.840740073, 0.999840856, 0.0162224691, 0.999998569, 0.995156705], [1.94203267e-06, 9.07052981e-06, 0.00239197793, 1.0, 0.0326851793, 0.999988198, 0.967085958, 0.668424819, 0.999859691, 0.011364375, 7.98012707e-07, 9.09890423e-06], [6.48978084e-06, 0.000158510258, 0.902566075, 0.000111232468, 0.999999523, 0.23738125, 0.743355691, 0.785282302, 0.999998927, 0.0100570468, 0.000157463131, 0.000314008066], [0.999985099, 0.424949557, 0.0407023691, 0.945959151, 1.23300947e-06, 1.60281015e-05, 6.80229959e-07, 0.848801288, 0.999905467, 0.0137382305, 0.999999881, 0.999975204], [0.999999881, 0.992633879, 0.985849738, 0.952115953, 0.000314034114, 0.00686225947, 3.9435592e-05, 0.968213589, 0.999869704, 0.0132045919, 0.999953866, 0.999974489], [0.999999881, 0.999835014, 0.0018423365, 0.000572863733, 2.58764589e-06, 5.82842076e-05, 1.25134954e-06, 0.953071585, 0.999946952, 0.0113394954, 0.999999404, 0.99917233], [0.00382398604, 0.000236873631, 0.999999881, 0.897863567, 0.999998331, 0.909701407, 0.0114350738, 0.960678619, 0.999999642, 0.994748072, 0.929442883, 0.119259201], [0.999355614, 0.999067605, 0.11367593, 0.00427671568, 6.29116221e-06, 1.46324865e-05, 0.000214812142, 0.975284036, 0.999990225, 0.0143781866, 0.999999762, 0.999999881], [0.000225597367, 0.048428122, 0.999989867, 0.0284612235, 0.999977946, 0.999606907, 0.999938965, 0.544306251, 0.999999285, 0.975159533, 0.000697045529, 3.1112595e-05], [0.996543586, 0.00359385181, 0.961288512, 9.43957129e-05, 6.32349256e-05, 3.86637948e-05, 1.04372448e-05, 0.938959458, 0.99870193, 0.00741781353, 0.00213561743, 2.4536987e-06], [6.73110299e-06, 0.000111849797, 0.00037034243, 8.61038789e-05, 0.000577011437, 0.00021719333, 0.999654531, 0.03415848, 0.00392012438, 0.0656879105, 0.886425972, 0.00131088099], [2.17288925e-05, 0.999980211, 1.89277107e-05, 1.48892491e-06, 0.000866502291, 0.00185316789, 0.000188228922, 0.00502494529, 2.39487576e-06, 0.0130146676, 0.99996078, 0.000450808438], [0.99999547, 0.0369585082, 0.774819016, 0.00129519799, 0.000718561991, 0.00171531993, 0.0936557055, 0.958816554, 0.999999762, 0.0136412376, 0.999999762, 0.999990106], [5.19536043e-05, 0.997177601, 0.999738514, 0.00015122164, 0.00203162408, 0.0535383336, 0.999893785, 0.975658224, 0.999999523, 0.981221791, 0.999859571, 0.035509441], [0.999999285, 0.0230828058, 0.999989033, 0.0013438008, 7.32175886e-06, 4.05365099e-05, 1.13602835e-06, 0.874401003, 0.992450237, 0.316143426, 0.999999881, 0.999716222], [1.90951978e-05, 0.0181415733, 0.000152718378, 1.84581459e-05, 0.000195690547, 0.000106842926, 0.00201702886, 0.920959979, 0.999928355, 0.00701835605, 0.999948978, 0.0143172797], [9.31491304e-05, 0.00949424319, 0.999986768, 0.00169689569, 0.99998486, 0.99995482, 1.0, 0.0180953461, 0.00127938914, 0.147758664, 0.000476803049, 2.38297434e-05], [0.999999166, 0.546855807, 1.0, 0.999999762, 0.925621033, 0.994842112, 0.0146205807, 0.430884121, 0.999999046, 0.0219339262, 0.000325489207, 4.18610143e-05], [0.999984622, 0.00153957936, 0.999999523, 0.177962914, 3.01406144e-05, 6.04498518e-06, 4.58740487e-06, 0.618366043, 0.187669173, 0.259949249, 0.962068677, 0.000313958066], [0.999968171, 0.215878904, 0.999999881, 0.999999881, 0.999999642, 0.999999881, 0.999979258, 0.947487097, 0.999976635, 0.960708969, 0.00111961924, 0.000825740513], [0.000873363111, 0.00344958389, 0.705652058, 0.00128735392, 0.48764047, 0.926094949, 0.999999642, 0.783250633, 0.99998343, 0.804306592, 0.990072131, 0.999981761], [1.84853798e-06, 1.17475693e-05, 0.000189696992, 6.15568424e-05, 8.03747025e-06, 9.00694977e-06, 1.80706793e-05, 0.0269404186, 0.000449639163, 0.0819870794, 1.01572505e-05, 4.78392622e-07], [0.637230515, 0.0387596935, 0.999988675, 0.0133497138, 0.00986258313, 0.977701008, 0.219596952, 0.932286715, 0.999985218, 0.819734929, 0.999931097, 0.65725106]], 'raptor384': [[0.4017189, 0.35853708, 0.21390659, 0.79821444, 0.2540394, 0.3356709, 0.25072342, 0.8344146, 0.5520537, 0.19038415, 0.36969236, 0.45180562], [0.34102768, 0.44412798, 0.13609937, 0.78239554, 0.36144224, 0.64267325, 0.86776847, 0.7751676, 0.7165401, 0.25172222, 0.8722955, 0.6430092], [0.6209574, 0.5782602, 0.953866, 0.75223064, 0.95148444, 0.8728104, 0.8566443, 0.7015383, 0.57890475, 0.58613396, 0.55284977, 0.5221764], [0.5285869, 0.40529218, 0.26047695, 0.3612213, 0.5871084, 0.4882787, 0.42861205, 0.80485743, 0.7232237, 0.28260696, 0.53845286, 0.47382736], [0.9703422, 0.4360616, 0.9186451, 0.42980164, 0.20184892, 0.28365168, 0.119350374, 0.37544024, 0.3950103, 0.246092, 0.41653198, 0.4488709], [0.48668984, 0.70299846, 0.31326395, 0.49505675, 0.43494445, 0.3752702, 0.4089355, 0.8858193, 0.8494772, 0.3881064, 0.90749097, 0.7660475], [0.9390626, 0.31020376, 0.080343544, 0.55850816, 0.2824902, 0.38726664, 0.17896461, 0.866956, 0.5458956, 0.18222377, 0.67506146, 0.58125687], [0.9039122, 0.9365863, 0.3810553, 0.24974501, 0.28959846, 0.32478014, 0.2803905, 0.49011105, 0.4984745, 0.41718125, 0.95602334, 0.8098695], [0.56610453, 0.6974681, 0.81070817, 0.9141061, 0.83502674, 0.7830041, 0.82980466, 0.830011, 0.66913843, 0.5436592, 0.844847, 0.9530119], [0.30779546, 0.45011115, 0.3364883, 0.9685844, 0.7831602, 0.97416085, 0.8593528, 0.89417577, 0.8278583, 0.9656784, 0.38306105, 0.4657526], [0.29217666, 0.3229539, 0.1850622, 0.20538095, 0.17552051, 0.2502281, 0.39239803, 0.375673, 0.4118078, 0.27322194, 0.6829014, 0.58666384], [0.90538406, 0.87574625, 0.6223094, 0.57714504, 0.19507678, 0.27516347, 0.2245325, 0.90977263, 0.6401347, 0.31877333, 0.939816, 0.7632382], [0.96181434, 0.7287357, 0.22382335, 0.8937781, 0.2501082, 0.2754767, 0.16690226, 0.71595615, 0.5550516, 0.24855196, 0.51293147, 0.42441747], [0.5933254, 0.4351588, 0.96564937, 0.82642853, 0.91081667, 0.6399406, 0.41925132, 0.8993275, 0.77386165, 0.98476064, 0.76392144, 0.777776], [0.7173064, 0.32373357, 0.15102598, 0.26391032, 0.14487824, 0.2347251, 0.12872371, 0.28598326, 0.31434342, 0.22780761, 0.83937037, 0.667216], [0.67325187, 0.43471795, 0.12705834, 0.8439058, 0.2671486, 0.51097095, 0.20283657, 0.5260008, 0.4604631, 0.33661556, 0.9530531, 0.9562892], [0.780275, 0.5690696, 0.61192393, 0.97723, 0.9362337, 0.9910866, 0.86156523, 0.88279396, 0.7962346, 0.9573097, 0.48401868, 0.3893462], [0.5261588, 0.32483616, 0.73062766, 0.9764108, 0.84074557, 0.95069444, 0.9393458, 0.7554807, 0.6611382, 0.28564346, 0.3612051, 0.46334743], [0.24144554, 0.57946646, 0.93672234, 0.9710657, 0.96065706, 0.7820139, 0.80999064, 0.933043, 0.72505665, 0.92439854, 0.77260816, 0.8712584], [0.97231454, 0.48584935, 0.13099106, 0.44772097, 0.15283222, 0.23489487, 0.12601307, 0.8868497, 0.55417323, 0.35075927, 0.9794244, 0.71953547], [0.9850597, 0.895335, 0.6227566, 0.77039576, 0.18950391, 0.38807583, 0.15992676, 0.82720184, 0.5482077, 0.23609138, 0.9812954, 0.78146434], [0.8911117, 0.37579364, 0.20365986, 0.90766084, 0.65255207, 0.8582463, 0.6570065, 0.6310034, 0.50737333, 0.97780526, 0.54997796, 0.4995137], [0.6119008, 0.4068085, 0.89815897, 0.45043224, 0.8020836, 0.5647037, 0.74083495, 0.838259, 0.6890459, 0.62772775, 0.5323746, 0.5120858], [0.9869627, 0.43691036, 0.22916916, 0.6011591, 0.06526211, 0.17855641, 0.060339473, 0.6882546, 0.45141578, 0.28047705, 0.9341347, 0.49703944], [0.26691365, 0.31145698, 0.96140575, 0.20409141, 0.6454078, 0.58849776, 0.42470503, 0.09276629, 0.24957857, 0.17451951, 0.33179992, 0.45928395], [0.6658697, 0.70414495, 0.9476463, 0.8524969, 0.97424424, 0.6408974, 0.6873586, 0.7828046, 0.68778145, 0.9832791, 0.60388196, 0.52215433], [0.8952526, 0.38282675, 0.15478045, 0.11448118, 0.09804882, 0.097417764, 0.08244601, 0.40470755, 0.3696495, 0.2839216, 0.9401989, 0.6600988], [0.96342504, 0.7327645, 0.8021904, 0.8712587, 0.32964516, 0.3557974, 0.21076259, 0.9137982, 0.6696644, 0.28225318, 0.9536276, 0.81598103], [0.8774034, 0.7939989, 0.7020456, 0.5471642, 0.20876473, 0.2908641, 0.24481028, 0.8927208, 0.70823324, 0.3519734, 0.98154926, 0.96589327], [0.9695582, 0.776387, 0.76066005, 0.80932945, 0.29389578, 0.26233315, 0.13997933, 0.9735428, 0.73930156, 0.32566896, 0.9908759, 0.9120084], [0.26659337, 0.4150325, 0.21648747, 0.082267396, 0.25420082, 0.35127887, 0.35097867, 0.3917604, 0.36310172, 0.7138579, 0.97382224, 0.8148726], [0.28016493, 0.23571086, 0.38107258, 0.9864086, 0.8594901, 0.98094165, 0.72191334, 0.93392766, 0.8674256, 0.26317585, 0.44688225, 0.60351074], [0.97738796, 0.79577833, 0.97975636, 0.5187574, 0.31723458, 0.27668852, 0.102623746, 0.81393266, 0.61770064, 0.30035615, 0.9444229, 0.5873135], [0.2777413, 0.48917624, 0.8987538, 0.42127427, 0.71956444, 0.32054248, 0.24595071, 0.8254356, 0.5808064, 0.35272568, 0.90846384, 0.87276167], [0.19931802, 0.7165816, 0.57170486, 0.20505303, 0.608806, 0.37898147, 0.555534, 0.506809, 0.46141565, 0.49214458, 0.2506026, 0.4402972], [0.9042939, 0.7708054, 0.47504717, 0.20099536, 0.5182359, 0.3626564, 0.33902484, 0.8056376, 0.5236994, 0.30599803, 0.94864124, 0.7887039], [0.23335797, 0.35575038, 0.6550826, 0.97326, 0.78359485, 0.9078226, 0.6870732, 0.69452095, 0.7232132, 0.26324725, 0.25814885, 0.4772638], [0.24870332, 0.39784694, 0.42342788, 0.4534793, 0.90194523, 0.67409563, 0.7512469, 0.8046948, 0.76676786, 0.20990238, 0.5612577, 0.5073292], [0.94583523, 0.8020115, 0.41154605, 0.64621377, 0.21707079, 0.26195738, 0.14410567, 0.6753117, 0.4757817, 0.2966193, 0.9834346, 0.90156454], [0.8687562, 0.89426255, 0.6822355, 0.59439504, 0.3538821, 0.32129592, 0.21764213, 0.7660158, 0.48151034, 0.25354016, 0.95601237, 0.87753356], [0.96885, 0.8125297, 0.162913, 0.31131858, 0.122287676, 0.23722118, 0.17138702, 0.9113096, 0.59616613, 0.34917474, 0.9779687, 0.6532098], [0.7860791, 0.5510839, 0.964123, 0.5825947, 0.9795089, 0.79482794, 0.65206933, 0.8543301, 0.6942262, 0.9749376, 0.75548583, 0.65770566], [0.9309987, 0.78426623, 0.49544942, 0.6514473, 0.2650876, 0.294294, 0.18310037, 0.9412325, 0.58741736, 0.31777734, 0.98358643, 0.96154207], [0.6058465, 0.6700704, 0.8686078, 0.4404465, 0.79924464, 0.6307899, 0.7090244, 0.860389, 0.7377702, 0.9792615, 0.53152055, 0.46112102], [0.9716227, 0.49214482, 0.8654544, 0.6827117, 0.31427762, 0.30127665, 0.13477823, 0.8166472, 0.5980972, 0.26738715, 0.38907564, 0.44109273], [0.32974482, 0.283103, 0.12363392, 0.21222438, 0.19937512, 0.2648235, 0.41161442, 0.29994643, 0.37094468, 0.24962613, 0.668479, 0.5382054], [0.25319695, 0.91519964, 0.13453862, 0.09567104, 0.33056536, 0.40573198, 0.5472739, 0.25034845, 0.31118917, 0.2676211, 0.91372776, 0.6594676], [0.99076855, 0.76119745, 0.53722894, 0.47998798, 0.25470904, 0.3479411, 0.2293373, 0.9629425, 0.76992, 0.29628804, 0.9772862, 0.79478705], [0.4025293, 0.73167753, 0.9162436, 0.6323475, 0.54662025, 0.67724156, 0.65790236, 0.9281311, 0.80860794, 0.95549744, 0.83630645, 0.66983485], [0.96479857, 0.56192553, 0.74899304, 0.3019109, 0.17169878, 0.18072206, 0.11306292, 0.85587347, 0.5173314, 0.82946044, 0.98445344, 0.71930134], [0.23897046, 0.5410076, 0.16015464, 0.08769807, 0.15047282, 0.1734413, 0.23961525, 0.86044145, 0.6320071, 0.17617178, 0.9424531, 0.6267232], [0.44267374, 0.43795806, 0.8935931, 0.40912694, 0.92309785, 0.8579704, 0.983793, 0.5381719, 0.5070294, 0.61045885, 0.42495736, 0.40240294], [0.9926725, 0.59124976, 0.9835892, 0.9673172, 0.780682, 0.8923175, 0.19550627, 0.7871127, 0.64277494, 0.32603717, 0.48493809, 0.45437005], [0.97305304, 0.48484272, 0.9512924, 0.5787482, 0.37637204, 0.33518052, 0.19153777, 0.8235937, 0.55202514, 0.44493282, 0.77403367, 0.6783869], [0.98539627, 0.6241182, 0.9753086, 0.97380507, 0.99296665, 0.99420595, 0.8074005, 0.9765227, 0.89424336, 0.7863622, 0.65574783, 0.6011328], [0.34868774, 0.4600789, 0.45598757, 0.5460267, 0.6316825, 0.787507, 0.9612317, 0.9008398, 0.74508965, 0.7064769, 0.55946183, 0.541139], [0.25051844, 0.25847316, 0.18755935, 0.1342926, 0.19373202, 0.09855336, 0.24544877, 0.35068393, 0.37324136, 0.43047005, 0.12955219, 0.36657798], [0.46739578, 0.62538326, 0.78116834, 0.6062794, 0.69119966, 0.8492689, 0.7025552, 0.8184029, 0.67435104, 0.8357619, 0.8585429, 0.69962204]], 'proxy29B': [[0.18965517, 0.1637931, 0.13793103, 0.55172414, 0.39655173, 0.3275862, 0.4051724, 0.5344828, 0.3448276, 0.13793103, 0.077586204, 0.094827585], [0.2672414, 0.37931034, 0.20689656, 0.73275864, 0.54310346, 0.69827586, 0.91810346, 0.47413793, 0.87068963, 0.18103448, 0.6034483, 0.54310346], [0.5086207, 0.5086207, 0.88793105, 0.6551724, 0.9439655, 0.8965517, 0.87931037, 0.27586207, 0.38793105, 0.6637931, 0.2413793, 0.3448276], [0.4224138, 0.25, 0.1637931, 0.19827586, 0.51724136, 0.51724136, 0.54310346, 0.37068966, 0.54310346, 0.36206895, 0.18103448, 0.13793103], [0.9051724, 0.49137932, 0.80172414, 0.49137932, 0.25, 0.29310346, 0.112068966, 0.06896552, 0.14655173, 0.077586204, 0.25, 0.15517241], [0.45689654, 0.7241379, 0.3275862, 0.55172414, 0.45689654, 0.39655173, 0.4827586, 0.62068963, 0.9741379, 0.37931034, 0.69827586, 0.67241377], [0.7844828, 0.18965517, 0.13793103, 0.5, 0.46551725, 0.6465517, 0.25, 0.82758623, 0.5344828, 0.060344826, 0.4224138, 0.41379312], [0.61206895, 0.9913793, 0.25862068, 0.13793103, 0.45689654, 0.43965518, 0.51724136, 0.21551724, 0.38793105, 0.6034483, 0.8362069, 0.86206895], [0.37068966, 0.55172414, 0.62931037, 0.8189655, 0.8448276, 0.80172414, 0.8836207, 0.5, 0.4224138, 0.7155172, 0.57758623, 0.94827586], [0.12068965, 0.30172414, 0.29310346, 0.86206895, 0.75, 0.95689654, 0.94827586, 0.80172414, 0.87068963, 0.94827586, 0.12931034, 0.25862068], [0.23275863, 0.14655173, 0.2413793, 0.1637931, 0.2413793, 0.18965517, 0.62931037, 0.1637931, 0.1724138, 0.4051724, 0.41379312, 0.37068966], [0.6810345, 0.94827586, 0.5948276, 0.5603448, 0.25862068, 0.29310346, 0.4224138, 0.6810345, 0.49137932, 0.6637931, 0.67241377, 0.73275864], [0.76724136, 0.76724136, 0.31034482, 0.8362069, 0.2672414, 0.29310346, 0.2672414, 0.21551724, 0.49137932, 0.10344828, 0.30172414, 0.12068965], [0.30172414, 0.21551724, 0.9698276, 0.62931037, 0.8448276, 0.43965518, 0.37931034, 0.75, 0.7844828, 0.95689654, 0.5, 0.79310346], [0.44827586, 0.112068966, 0.12068965, 0.28448275, 0.0862069, 0.2413793, 0.12068965, 0.05172414, 0.04310345, 0.14655173, 0.49137932, 0.35344827], [0.41379312, 0.21551724, 0.05172414, 0.7586207, 0.29310346, 0.6034483, 0.37931034, 0.25862068, 0.2672414, 0.57758623, 0.69827586, 0.94827586], [0.57758623, 0.36206895, 0.5689655, 0.9655172, 0.95258623, 0.9741379, 0.92241377, 0.7241379, 0.86206895, 0.9137931, 0.14655173, 0.25], [0.25, 0.077586204, 0.70689654, 0.95689654, 0.8362069, 0.9310345, 0.94827586, 0.36206895, 0.5862069, 0.43103448, 0.060344826, 0.1724138], [0.28448275, 0.5, 0.86206895, 0.9094828, 0.95258623, 0.8189655, 0.9094828, 0.7844828, 0.8189655, 0.8448276, 0.44827586, 0.82758623], [0.87931037, 0.5, 0.0862069, 0.29310346, 0.094827585, 0.19827586, 0.0862069, 0.8189655, 0.4224138, 0.46551725, 0.86206895, 0.7155172], [0.9396552, 0.9396552, 0.57758623, 0.62931037, 0.1724138, 0.54310346, 0.18965517, 0.5, 0.31896552, 0.12068965, 0.8965517, 0.7413793], [0.55172414, 0.20689656, 0.20689656, 0.86206895, 0.6896552, 0.82758623, 0.6465517, 0.23275863, 0.20689656, 0.9741379, 0.37931034, 0.25], [0.28448275, 0.27586207, 0.7586207, 0.5862069, 0.76724136, 0.55172414, 0.73275864, 0.6465517, 0.6465517, 0.73275864, 0.2413793, 0.36206895], [0.88793105, 0.35344827, 0.21551724, 0.3448276, 0.02586207, 0.05172414, 0.01724138, 0.30172414, 0.20689656, 0.30172414, 0.70689654, 0.37931034], [0.18103448, 0.0862069, 0.86206895, 0.13793103, 0.67241377, 0.6465517, 0.62068963, 0.01724138, 0.04310345, 0.13793103, 0.12068965, 0.1637931], [0.44827586, 0.7241379, 0.87068963, 0.67241377, 0.9784483, 0.76724136, 0.79310346, 0.37931034, 0.51724136, 0.9655172, 0.3448276, 0.30172414], [0.6896552, 0.2672414, 0.27586207, 0.06896552, 0.05172414, 0.01724138, 0.04310345, 0.13793103, 0.06896552, 0.29310346, 0.62931037, 0.5603448], [0.8362069, 0.70689654, 0.7241379, 0.8362069, 0.37931034, 0.51724136, 0.30172414, 0.88793105, 0.70689654, 0.21551724, 0.87931037, 0.86206895], [0.6465517, 0.87068963, 0.5, 0.33620688, 0.18103448, 0.1724138, 0.5, 0.88793105, 0.7758621, 0.5862069, 0.87068963, 0.9913793], [0.8103448, 0.8362069, 0.79741377, 0.76724136, 0.31034482, 0.1724138, 0.2672414, 0.88793105, 0.79310346, 0.38793105, 0.98275864, 0.94827586], [0.28448275, 0.39655173, 0.21551724, 0.1724138, 0.35344827, 0.5344828, 0.47413793, 0.14655173, 0.077586204, 0.7758621, 0.9051724, 0.7586207], [0.18965517, 0.03448276, 0.22413793, 0.99568963, 0.73275864, 0.9655172, 0.80172414, 0.9310345, 0.9396552, 0.28448275, 0.25862068, 0.5344828], [0.88793105, 0.9051724, 0.9612069, 0.5086207, 0.5603448, 0.4224138, 0.13793103, 0.33620688, 0.49137932, 0.29310346, 0.75, 0.54310346], [0.29310346, 0.57758623, 0.8103448, 0.1724138, 0.75, 0.44827586, 0.5689655, 0.57758623, 0.55172414, 0.62068963, 0.6034483, 0.82758623], [0.02586207, 0.76724136, 0.4827586, 0.18103448, 0.63793105, 0.49137932, 0.5862069, 0.23275863, 0.20689656, 0.61206895, 0.06896552, 0.0862069], [0.75, 0.79310346, 0.44827586, 0.060344826, 0.57758623, 0.43965518, 0.4051724, 0.5258621, 0.39655173, 0.47413793, 0.7758621, 0.7155172], [0.05172414, 0.094827585, 0.47413793, 0.9612069, 0.69827586, 0.9051724, 0.73275864, 0.38793105, 0.62068963, 0.28448275, 0.03448276, 0.18965517], [0.112068966, 0.25, 0.47413793, 0.33620688, 0.88793105, 0.69827586, 0.76724136, 0.49137932, 0.86206895, 0.1637931, 0.27586207, 0.33620688], [0.73275864, 0.80172414, 0.41379312, 0.6637931, 0.13793103, 0.14655173, 0.10344828, 0.43965518, 0.37931034, 0.43103448, 0.95689654, 0.87068963], [0.7887931, 0.88793105, 0.5862069, 0.63793105, 0.45689654, 0.43965518, 0.31896552, 0.63793105, 0.37068966, 0.31034482, 0.7586207, 0.8534483], [0.9008621, 0.9310345, 0.2672414, 0.29310346, 0.05172414, 0.1637931, 0.1637931, 0.8448276, 0.5603448, 0.43103448, 0.87068963, 0.6034483], [0.54310346, 0.43103448, 0.9353448, 0.5948276, 0.9396552, 0.7586207, 0.6034483, 0.75, 0.82758623, 0.94827586, 0.47413793, 0.55172414], [0.67241377, 0.87068963, 0.47413793, 0.55172414, 0.2413793, 0.20689656, 0.3275862, 0.95689654, 0.61206895, 0.46551725, 0.9353448, 0.9913793], [0.41379312, 0.63793105, 0.73275864, 0.43103448, 0.80172414, 0.73275864, 0.7844828, 0.5344828, 0.86206895, 0.92241377, 0.25862068, 0.21551724], [0.7413793, 0.4827586, 0.6465517, 0.46551725, 0.37068966, 0.23275863, 0.1724138, 0.62931037, 0.45689654, 0.1724138, 0.21551724, 0.0862069], [0.19827586, 0.14655173, 0.15517241, 0.21551724, 0.3275862, 0.2672414, 0.6637931, 0.12068965, 0.13793103, 0.41379312, 0.43965518, 0.41379312], [0.18103448, 0.9913793, 0.0862069, 0.04310345, 0.4827586, 0.5, 0.5, 0.05172414, 0.02586207, 0.33620688, 0.6896552, 0.47413793], [0.87068963, 0.6896552, 0.5, 0.4051724, 0.39655173, 0.4224138, 0.5, 0.9051724, 0.92241377, 0.41379312, 0.8836207, 0.8448276], [0.30172414, 0.8103448, 0.7241379, 0.45689654, 0.5603448, 0.6896552, 0.7413793, 0.9396552, 0.9310345, 0.88793105, 0.5862069, 0.5689655], [0.86206895, 0.55172414, 0.67241377, 0.3275862, 0.13793103, 0.12931034, 0.077586204, 0.6465517, 0.3448276, 0.80172414, 0.9741379, 0.6896552], [0.13793103, 0.51724136, 0.18103448, 0.06896552, 0.23275863, 0.15517241, 0.44827586, 0.69827586, 0.5689655, 0.03448276, 0.70689654, 0.5], [0.33620688, 0.43103448, 0.7155172, 0.36206895, 0.87931037, 0.8534483, 1.0, 0.15517241, 0.20689656, 0.7155172, 0.20689656, 0.13793103], [0.95689654, 0.6637931, 0.99568963, 0.89224136, 0.75, 0.8362069, 0.43103448, 0.37931034, 0.73275864, 0.5344828, 0.22413793, 0.21551724], [0.80172414, 0.41379312, 0.86206895, 0.5603448, 0.38793105, 0.23275863, 0.21551724, 0.49137932, 0.31896552, 0.70689654, 0.5086207, 0.49137932], [0.80172414, 0.6465517, 0.95258623, 0.94827586, 0.9655172, 1.0, 0.8362069, 0.88793105, 0.82758623, 0.8362069, 0.33620688, 0.44827586], [0.37068966, 0.43965518, 0.46551725, 0.4224138, 0.6810345, 0.7586207, 0.9698276, 0.6810345, 0.76724136, 0.7844828, 0.43965518, 0.62931037], [0.077586204, 0.05172414, 0.21551724, 0.13793103, 0.1724138, 0.060344826, 0.33620688, 0.112068966, 0.112068966, 0.6637931, 0.06896552, 0.01724138], [0.46551725, 0.62068963, 0.6810345, 0.5344828, 0.6465517, 0.7844828, 0.7155172, 0.62931037, 0.6810345, 0.82758623, 0.62931037, 0.62931037]]}
S6_SOURCE = 'MODE = "submit"\n\nK1_BUILD_ID = "2026-10-12T10:12:34+03:00/f0d87208a3de20a5ac936388bfd763799d6391833fea3a4f90ec6de14ef5051b"\n\nimport base64, faulthandler, hashlib, json, math, os, re, shutil, signal, subprocess, sys, tarfile, threading, time, traceback\n\nfrom pathlib import Path\n\nimport numpy as np\n\nimport pandas as pd\n\nKROOT = Path(os.environ.get("K1_KAGGLE_ROOT", "/kaggle"))\n\nINPUT, WORK = KROOT / "input", KROOT / "working"\n\nOUT = WORK / "k1"\n\nTMP = Path(os.environ.get("K1_TMP", "/tmp/k1"))\n\nCOMP = "rsna-knee-abnormality-detection"\n\nDS_SLUG = "rsna-knee-s6fall-weights"\n\nHELIOS_SLUG = "rsna-orthofoundation-fold0-epoch7-20261004"\n\nEXTRAS_FILE = "s6_extras.bin"\n\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n       "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n\nW_A, W_B = 0.0, 1.0\n\nA_SOURCE_SHA256 = "bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5"\n\nA_PUBLIC_CSV_SHA256 = "faa20e95a1779658237252388407eaf6d835679dd43e4dae5bca18d95496bede"\n\nPUBLIC_TEST_IDS = ["1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191",\n                   "1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776",\n                   "1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402"]\n\nA_FALLBACK_MIN, A_FALLBACK_FRAC = 3, 0.01\n\nB_MODEL = {"file": "E_prior_s42_fall.pt", "source_file": "E_prior_s42_fall.pt", "sha256": "0c7297d58f061ab4e5d7c55046f51fa3e8934928046f0647a0c7854e98371c12", "run_config": "E_prior_s42_fall.run_config.json", "done": "E_prior_s42_fall.DONE.json", "arm": "E_prior", "expect_epochs": 16, "expect_train": 4263}\n\nANATOMY_FILE, ANATOMY_SHA256 = "anatomy.pt", "5904dd16eb9a4fd7a1caf90ac977f8fe2ef364908598c23c209db3fe7fd1563e"\n\nPAYLOAD_FILE, PAYLOAD_MAGIC = "k1_code_payload.bin", b"K1PAYLOAD1\\n"\n\nSOURCE_SHA = {                                                 # the 0.954 notebook\'s SOURCE_SHA = S6F0/s6common.SOURCE_SHA\n    "px_cache.py": "b13e38a7ad3bd496f8027e02209653e224b0926db4f0853eeb0d076a2d6e2632",\n    "compact_train.py": "14767c7a8f0daa986477511fe9a83aefe032a208ae978fabf15a5f896992ac80",\n    "compact_infer.py": "34f433fe72b732543813165c3397ec80cef01b0d49b25b7b22668714f0b0ef4a",\n    "probe_infer.py": "a6cc31448019c84be5868d0a2ff1d99719b23b0f86110efc03efff3ce6bd6980",\n    "knee_anchor_bank_v2.py": "e564807a129dfcd3c984348917436ee6083066a4d685340587ebd1d3ea14f309",\n    "side_common.py": "bbcc5d027834e6a6fd48c4a8cf228e25661b7522c792587c840fe4d7cd5c0566",\n}\n\nB_RECEIPT_SOURCES = ("px_cache.py", "compact_train.py", "compact_infer.py", "probe_infer.py", "knee_anchor_bank_v2.py")\n\nCODE_SHA = {                                                   # the 0.954 notebook\'s CODE_SHA (cloud_code/training pins)\n    "native_cache.py": "c53ce8985406b3b918d9e821d43518479e89d9f946daebd817f378e3949718b9",\n    "native_data.py": "f59afe205c8e193369fa3ccd22edcce8df3b293205fc48ddb1dd45169ee43b36",\n    "input_provenance.py": "b3a22500c7f976f6ee040239ada5cacecf2326861cb49af1c7d8cfdeeb55fcda",\n    "intensity_mapping.py": "19be2d8cbd6f8971c1288d7512a8cca41d143470c2ba6e9b9096fde035fa9d41",\n    "coronal_views.py": "841f216d2682ae2c0f8afd82fc902b9fa74924d7145b952069dea8b0bbd03c40",\n    "exact_duplicates.py": "51b38db21ce9c5fe2b6278a835983414756b61ac6af81aba380b44b41b075897",\n    "dicom_geometry.py": "12aa501f9704aef15b7a3bbd560951acb6d95b89b7de0141d6cf89fc3f051ab1",\n    "common.py": "8283fa1f8e7ef8cfacbba52df1715636be3c9ad688e39912814e7d5bcf57105f",\n}\n\nB_GEOMETRY = {"backbone": "convnext_small", "size": 320, "crop": 288, "views": "v4", "slices": 3, "phys_mm": 3.5,\n              "eval_stride": 1, "chunk": 192, "item_timeout": 900}\n\nB_AMP = "fp16"\n\nB_WORKERS = 2\n\nMIN_RHO_MEAN, MIN_RHO_FINDING, RHO_MIN_N = 0.60, 0.40, 100\n\nB_FAIL_MIN, B_FAIL_FRAC = 3, 0.0025\n\nA_TIMEOUT_S = 9000\n\nB_TIMEOUT_S = 14400\n\nPARITY_TIMEOUT_S = 3600\n\nGUARD_MAX_S = 480 * 60\n\nSETUP_SLACK_S = 30 * 60\n\nEND_SLACK_S = 10 * 60\n\nA_STALL_S = 30 * 60\n\nA_RETRY_MIN_S = 45 * 60\n\nEXIT_GRACE_S = 120\n\nB_DEADLINE_MARGIN_S = 20 * 60\n\nB_RETRY_MIN_S = 10 * 60\n\nGUARD_S = GUARD_MAX_S + (PARITY_TIMEOUT_S if MODE == "parity" else 0)\n\nassert A_TIMEOUT_S + B_TIMEOUT_S + SETUP_SLACK_S <= GUARD_MAX_S, "budgets exceed the hang guard"\n\nBACKSTOP_EXTRA_S = 10 * 60\n\nHEARTBEAT_S = 300\n\nPARITY_IDS = []\n\nT0 = time.time()\n\n_LOG = None\n\n_GUARD_FILE = None\n\n_CHILDREN = {}\n\n_GUARD_STOP = threading.Event()\n\n_RUN = {"nonce": os.urandom(16).hex(), "started_utc": time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime(T0)),\n        "kaggle_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE")}\n\n_RECEIPT = {"mode": MODE, "build_id": K1_BUILD_ID, "run": _RUN, "stages": {}, "gpu": {}}\n\nclass K1Error(RuntimeError):\n    pass\n\nclass K1Timeout(K1Error):\n    """a stage hit its time cap, or (A attempt 1) its log stalled"""\n\ndef log(*a):\n    msg = "%s +%6.0fs %s" % (time.strftime("%H:%M:%S", time.gmtime()), time.time() - T0, " ".join(str(x) for x in a))\n    print(msg, flush=True)\n    if _LOG is not None:\n        try:\n            _LOG.write(msg + "\\n"); _LOG.flush()\n        except Exception:\n            pass\n\ndef sha256_file(p):\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\ndef build_id_of(text):\n    """K1_BUILD_ID recomputed from a run.py text (= build_k1.build_id_of = compare_parity.build_id_of): \'<at>/<sha256(at + newline\n    + the text with the constant\'s value put back to the template placeholder)>\'; None if the text has no built id line."""\n    m = re.search(r\'^K1_BUILD_ID = "([^"/]+)/([0-9a-f]{64})"\', text, re.M)\n    if m is None:\n        return None\n    base = text[:m.start()] + \'K1_BUILD_ID = "__K1_\' + \'BUILD_ID__"\' + text[m.end():]     # split: the template holds one placeholder\n    return m.group(1) + "/" + hashlib.sha256((m.group(1) + "\\n" + base).encode()).hexdigest()\n\ndef self_identity():\n    """This execution\'s own source, when the runner exposes it (Kaggle runs a script as /kaggle/src/script.py; a notebook-style\n    wrapper may define no __file__): its path, sha256 and the build id recomputed from its text (derived_matches: that id equals\n    the baked K1_BUILD_ID).  Only a file holding this build\'s K1_BUILD_ID line counts.  Never raises; None when unavailable."""\n    out = {"path": None, "sha256": None, "derived_build_id": None, "derived_matches": None}\n    for cand in (globals().get("__file__"), sys.argv[0] if sys.argv else None):\n        try:\n            if not cand or not os.path.isfile(cand):\n                continue\n            text = Path(cand).read_text(errors="replace")\n            if (\'K1_BUILD_ID = "%s"\' % K1_BUILD_ID) not in text:\n                continue\n            out.update(path=str(cand), sha256=sha256_file(cand), derived_build_id=build_id_of(text))\n            out["derived_matches"] = out["derived_build_id"] == K1_BUILD_ID\n            break\n        except Exception as e:\n            out["error"] = repr(e)[:200]\n    return out\n\ndef save_json(path, obj):\n    path = Path(path); tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=1, default=str)); os.replace(tmp, path)\n\ndef tail(path, n=3000):\n    try:\n        return Path(path).read_text(errors="replace")[-n:]\n    except Exception as e:\n        return "(no log: %r)" % e\n\ndef remove_submission():\n    for p in (WORK / "submission.csv", WORK / ".submission.csv.tmp"):\n        try:\n            p.unlink()\n        except FileNotFoundError:\n            pass\n\ndef gpu_snapshot():\n    try:\n        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],\n                           capture_output=True, text=True, timeout=20)\n        return [l.strip() for l in r.stdout.splitlines() if l.strip()] or [r.stderr.strip()[:200]]\n    except Exception as e:\n        return ["nvidia-smi failed: %r" % e]\n\ndef mem_available_mb():\n    try:\n        for line in open("/proc/meminfo"):\n            if line.startswith("MemAvailable:"):\n                return int(line.split()[1]) // 1024\n    except Exception:\n        pass\n    return -1\n\ndef group_rss_mb(pgid):\n    tot = 0\n    for d in os.listdir("/proc"):\n        if not d.isdigit():\n            continue\n        try:\n            st = open("/proc/%s/stat" % d).read()\n            f = st[st.rindex(")") + 2:].split()\n            if int(f[2]) == pgid:\n                tot += int(f[21]) * os.sysconf("SC_PAGE_SIZE")\n        except Exception:\n            continue\n    return tot // (1 << 20)\n\ndef kill_group(pgid):\n    try:\n        os.killpg(pgid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        pass\n    _CHILDREN.pop(pgid, None)\n\ndef kill_all_children():\n    for pgid in list(_CHILDREN):\n        kill_group(pgid)\n\ndef stage(name):\n    _RECEIPT["stages"][name] = {"start_s": round(time.time() - T0, 1)}\n    _RECEIPT["gpu"][name + ".start"] = gpu_snapshot()\n    log("[stage] %s start | gpu %s | MemAvailable %d MB" % (name, _RECEIPT["gpu"][name + ".start"], mem_available_mb()))\n\ndef stage_done(name, **info):\n    s = _RECEIPT["stages"][name]\n    s["seconds"] = round(time.time() - T0 - s["start_s"], 1)\n    s.update(info)\n    _RECEIPT["gpu"][name + ".end"] = gpu_snapshot()\n    log("[stage] %s done in %.0fs | gpu %s" % (name, s["seconds"], _RECEIPT["gpu"][name + ".end"]))\n\ndef _watchdog(cap_s):\n    deadline = T0 + cap_s\n    while True:\n        left = deadline - time.time()\n        if left <= 0:\n            break\n        if _GUARD_STOP.wait(min(left, 15)):\n            return\n    try:\n        n_children = len(_CHILDREN)\n        kill_all_children()                                  # cleanup first: a failing dump below must not skip it\n        remove_submission()\n        log("[guard] FIRED after %.0fs (cap %ds): killed %d child group(s), deleted submission.csv; dumping every thread"\n            % (time.time() - T0, cap_s, n_children))\n        for target in (_GUARD_FILE, sys.stdout):\n            try:\n                faulthandler.dump_traceback(file=target, all_threads=True)\n            except Exception as e:\n                log("[guard] dump to %r failed: %r" % (target, e))\n        try:\n            save_json(OUT / "k1_error.json", {"error": "hang guard fired", "cap_s": cap_s, "elapsed_s": round(time.time() - T0, 1),\n                                              "receipt": _RECEIPT})\n        except Exception:\n            pass\n        sys.stdout.flush()\n    finally:\n        os._exit(1)\n\ndef arm_guard(cap_s):\n    global _GUARD_FILE\n    _GUARD_FILE = open(OUT / "k1_hang_guard_stacks.txt", "w")\n    faulthandler.enable(file=_GUARD_FILE, all_threads=True)\n    faulthandler.dump_traceback_later(cap_s + BACKSTOP_EXTRA_S, repeat=False, file=_GUARD_FILE, exit=True)\n    threading.Thread(target=_watchdog, args=(cap_s,), name="k1-guard", daemon=True).start()\n    log("[guard] armed: watchdog %ds (kill children, delete submission.csv, _exit 1); faulthandler backstop %ds"\n        % (cap_s, cap_s + BACKSTOP_EXTRA_S))\n\ndef guard_left():\n    return T0 + GUARD_S - time.time()\n\ndef _size(path):\n    try:\n        return os.path.getsize(path)\n    except OSError:\n        return -1\n\ndef run_group(specs, timeout_s, what):\n    """Start every spec {name, cmd, env, cwd, log[, done_when, stall_s]} in its own session; poll until all exit, one fails, or\n    timeout_s passes.  A failure, a stall or the timeout kills every group (DataLoader / Pool children included).\n    done_when(): the child\'s outputs are complete; if it then has not exited within EXIT_GRACE_S (an interpreter-exit hang), its\n    group is killed and the run counts as a success.  stall_s: the log has not grown for that long = K1Timeout.\n    Returns [(name, rc, seconds, note)]."""\n    procs, t_start, last_hb, last_check = [], time.time(), time.time(), 0.0\n    try:\n        for s in specs:\n            fh = open(s["log"], "w")\n            p = subprocess.Popen(s["cmd"], env=s["env"], cwd=s.get("cwd"), stdout=fh, stderr=subprocess.STDOUT,\n                                 stdin=subprocess.DEVNULL, start_new_session=True)\n            _CHILDREN[p.pid] = s["name"]\n            procs.append({"s": s, "p": p, "fh": fh, "t1": time.time(), "size": -1, "changed": time.time(), "done_at": None})\n            log("[%s] started %s pid %d" % (what, s["name"], p.pid))\n        done = {}\n        while len(done) < len(procs):\n            now_ = time.time()\n            check = now_ - last_check >= 5\n            if check:\n                last_check = now_\n            for r in procs:\n                s, p = r["s"], r["p"]\n                if s["name"] in done:\n                    continue\n                rc, note = p.poll(), ""\n                if rc is None and check and s.get("done_when") is not None:\n                    try:\n                        complete = bool(s["done_when"]())\n                    except Exception:\n                        complete = False\n                    if complete and r["done_at"] is None:\n                        r["done_at"] = now_\n                    if r["done_at"] is not None and now_ - r["done_at"] > EXIT_GRACE_S and p.poll() is None:\n                        kill_group(p.pid)\n                        rc, note = 0, "outputs complete, no exit after %ds: group killed, accepted" % EXIT_GRACE_S\n                        log("[%s] %s: %s" % (what, s["name"], note))\n                if rc is not None:\n                    done[s["name"]] = (rc, round(time.time() - r["t1"], 1), note)\n                    r["fh"].close(); kill_group(p.pid)\n                    log("[%s] %s exited rc %d after %.0fs" % (what, s["name"], rc, time.time() - r["t1"]))\n                    if rc != 0:\n                        for r2 in procs:\n                            if r2["s"]["name"] not in done:\n                                kill_group(r2["p"].pid); log("[%s] killed %s (a sibling failed)" % (what, r2["s"]["name"]))\n                        raise K1Error("%s: %s exited %d; log tail:\\n%s" % (what, s["name"], rc, tail(s["log"])))\n                    continue\n                if check and s.get("stall_s"):\n                    sz = _size(s["log"])\n                    if sz != r["size"]:\n                        r["size"], r["changed"] = sz, now_\n                    elif now_ - r["changed"] > s["stall_s"]:\n                        for r2 in procs:\n                            if r2["s"]["name"] not in done:\n                                kill_group(r2["p"].pid)\n                        raise K1Timeout("%s: %s stalled: log unchanged for %ds (at %d B); log tail:\\n%s" % (\n                            what, s["name"], s["stall_s"], sz, tail(s["log"], 1500)))\n            if len(done) == len(procs):\n                break\n            if time.time() - t_start > timeout_s:\n                for r in procs:\n                    if r["s"]["name"] not in done:\n                        kill_group(r["p"].pid)\n                raise K1Timeout("%s: timeout after %ds; still running %s" % (\n                    what, timeout_s, [r["s"]["name"] for r in procs if r["s"]["name"] not in done]))\n            if time.time() - last_hb >= HEARTBEAT_S:\n                last_hb = time.time()\n                for r in procs:\n                    if r["s"]["name"] in done:\n                        continue\n                    prog = [l for l in tail(r["s"]["log"], 4000).splitlines() if l.startswith("{") or "[gpu]" in l or "[pre]" in l][-1:]\n                    log("[heartbeat] %s %s | %.0fs | rss %d MB | MemAvailable %d MB | gpu %s | %s" % (\n                        what, r["s"]["name"], time.time() - r["t1"], group_rss_mb(r["p"].pid), mem_available_mb(), gpu_snapshot(),\n                        prog[0][:300] if prog else "(no progress line yet)"))\n            time.sleep(2)\n        return [(r["s"]["name"],) + done[r["s"]["name"]] for r in procs]\n    finally:\n        for r in procs:\n            if r["p"].poll() is None:\n                kill_group(r["p"].pid)\n            try:\n                r["fh"].close()\n            except Exception:\n                pass\n\nPROBE = r\'\'\'\nimport json, sys, importlib.util\nout = {"python": sys.version.split()[0]}\nimport torch\nout.update(torch=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n           cuda_available=torch.cuda.is_available(), n_gpu=torch.cuda.device_count())\nif torch.cuda.is_available():\n    out["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n    out["capability"] = list(torch.cuda.get_device_capability(0))\n    out["bf16_supported"] = bool(torch.cuda.is_bf16_supported())\nfor m in ("numpy", "pandas", "timm", "pydicom", "cv2", "scipy", "safetensors"):\n    try:\n        out[m] = getattr(__import__(m), "__version__", "?")\n    except Exception as e:\n        out[m] = "MISSING " + type(e).__name__\nfor m in ("requests", "kagglesdk", "pylibjpeg", "gdcm"):\n    out["has_" + m] = importlib.util.find_spec(m) is not None\nprint("K1_ENV " + json.dumps(out), flush=True)\n\'\'\'\n\ndef probe_env():\n    r = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True, timeout=300, stdin=subprocess.DEVNULL)\n    line = [l for l in r.stdout.splitlines() if l.startswith("K1_ENV ")]\n    if r.returncode != 0 or not line:\n        raise K1Error("environment probe failed rc %d: %s %s" % (r.returncode, r.stdout[-1500:], r.stderr[-1500:]))\n    return json.loads(line[0][len("K1_ENV "):])\n\ndef find_mount(slug, marker):\n    cands = [INPUT / slug, INPUT / "competitions" / slug]\n    cands += list(INPUT.glob("datasets/*/" + slug)) + list(INPUT.glob("datasets/*/" + slug + "/*"))\n    cands += list(INPUT.glob("*/" + slug)) + list(INPUT.glob(slug + "/*"))\n    found = []\n    for p in cands:\n        if (p / marker).is_file() and p.resolve() not in [f.resolve() for f in found]:\n            found.append(p)\n    if len(found) != 1:\n        raise K1Error("cannot resolve %s (marker %s): %s" % (slug, marker, found))\n    return found[0]\n\ndef safe_rel(name):\n    p = Path(name)\n    if p.is_absolute() or ".." in p.parts or not p.parts:\n        raise K1Error("unsafe payload member %r" % name)\n    return p\n\ndef extract_payload(path, target, members):\n    """The code payload = PAYLOAD_MAGIC + an uncompressed tar (opaque to Kaggle\'s archive handling); exact member set, sizes, sha."""\n    if target.exists():\n        shutil.rmtree(target)\n    with open(path, "rb") as f:\n        if f.read(len(PAYLOAD_MAGIC)) != PAYLOAD_MAGIC:\n            raise K1Error("payload magic mismatch")\n        with tarfile.open(fileobj=f, mode="r:") as tar:\n            ms = tar.getmembers()\n            if {m.name for m in ms} != set(members) or len(ms) != len(members):\n                raise K1Error("payload member set differs from MANIFEST")\n            for m in ms:\n                safe_rel(m.name)\n                if not m.isfile() or m.size != members[m.name]["bytes"]:\n                    raise K1Error("payload member %s: not a regular file or size differs" % m.name)\n            for m in ms:\n                dst = target / safe_rel(m.name)\n                dst.parent.mkdir(parents=True, exist_ok=True)\n                with tar.extractfile(m) as src, open(dst, "xb") as out:\n                    shutil.copyfileobj(src, out, 1 << 20)\n    for name, rec in members.items():\n        if sha256_file(target / name) != rec["sha256"]:\n            raise K1Error("payload member %s sha differs" % name)\n\nLAUNCHER = r\'\'\'"""K1 component-B launcher = S6F0/infer_s6f0.py with Kaggle paths.  Runs Heliosli\'s compact_infer.main() UNCHANGED.\nAs on the laptop: sys.path = [cloud_code, heliosli, ...] and the training.download_data stub (only the chunked-archive reader\nuses it; we read the DICOM directory), installed in the main process AND in every spawn worker (CStudies._setup).\nThree K1 guards, none of which changes a prediction on the laptop\'s path (each acts only where that path would fail):\n- distutils: the vendored MONAI 0.6.0 imports distutils.util.strtobool (Python >= 3.12 has no stdlib distutils; setuptools\'\n  shim provides it on the laptop).  Only if that import fails, a module holding the stdlib\'s strtobool is installed.\n- setup failure (review M1): compact_infer\'s _setup sets self.native before it loads the SKM-TEA V-Net, so a failed load left a\n  worker scoring every later study with field-centre views and no anchor, silently.  Here a failed _setup resets self.native\n  (the next study re-runs it; probe_infer.install_native is memoised, the one step that is not re-entrant) and re-raises (this\n  study becomes a fallback, retried); a scored item without an anchor is turned into a failed item (fallback, retried).\nK1_PX_DUMP (parity mode only): every item\'s pixel / meta hashes are appended to <dir>/px_<pid>.jsonl; the item is not changed."""\nimport hashlib, json, os, sys, types\nSRC = __SRC__\nCODE = __CODE__\nANCHOR_VIEWS = ("v3k", "v4")\n\n\ndef _strtobool(val):\n    """distutils.util.strtobool (CPython 3.11 / setuptools._distutils), unchanged semantics."""\n    val = val.lower()\n    if val in ("y", "yes", "t", "true", "on", "1"):\n        return True\n    elif val in ("n", "no", "f", "false", "off", "0"):\n        return False\n    else:\n        raise ValueError(f"invalid truth value {val!r}")\n\n\ndef install_distutils():\n    if "distutils.util" in sys.modules:\n        return\n    try:\n        import distutils.util  # noqa: F401  (setuptools\' shim on Python >= 3.12)\n    except Exception:\n        d, u = types.ModuleType("distutils"), types.ModuleType("distutils.util")\n        d.__path__ = []\n        u.strtobool = _strtobool\n        d.util = u\n        sys.modules["distutils"], sys.modules["distutils.util"] = d, u\n        os.environ["K1_DISTUTILS_SHIM"] = "1"\n\n\ndef install_paths():\n    sys.dont_write_bytecode = True\n    for p in (SRC, CODE):\n        if p not in sys.path:\n            sys.path.insert(0, p)\n    if "training.download_data" not in sys.modules:\n        m = types.ModuleType("training.download_data")\n        m.ChunkedArchive = None\n        sys.modules["training.download_data"] = m\n    install_distutils()\n\n\ndef memoise_install_native():\n    import probe_infer as PI\n    if getattr(PI, "_k1_memo", False):\n        return\n    orig, cache = PI.install_native, {}\n\n    def install_native(lean=True):\n        if lean not in cache:\n            cache[lean] = orig(lean)\n        return cache[lean]\n    PI.install_native = install_native\n    PI._k1_memo = True\n\n\ninstall_paths()\nimport compact_infer as CI\n\nassert os.path.dirname(os.path.abspath(CI.__file__)) == SRC, CI.__file__\nPX_DUMP = os.environ.get("K1_PX_DUMP")\n\n\nclass LocalCStudies(CI.CStudies):\n    def _setup(self):\n        install_paths()\n        memoise_install_native()\n        try:\n            super()._setup()\n            if self.views in ANCHOR_VIEWS and self.bank is None:\n                raise RuntimeError("K1: the SKM-TEA anchor bank did not load")\n        except BaseException:\n            self.native = None\n            raise\n\n    def __getitem__(self, i):\n        out = super().__getitem__(i)\n        if out.get("error") is None and self.views in ANCHOR_VIEWS and out.get("anchor") is None:\n            out = dict(out, error="K1: item built without an SKM-TEA anchor", images=out["images"][:0], meta=out["meta"][:0])\n        if PX_DUMP:\n            try:\n                img = out["images"].numpy()\n                meta = out["meta"].numpy()\n                rec = {"uid": out["uid"], "error": out["error"], "n": int(len(img)),\n                       "img_sha256": hashlib.sha256(img.tobytes()).hexdigest(),\n                       "meta_sha256": hashlib.sha256(meta.astype("float32").tobytes()).hexdigest(),\n                       "win_sha1_10": [hashlib.sha1(w.tobytes()).hexdigest()[:10] for w in img],\n                       "img_mean": float(img.mean()) if len(img) else None, "anchor": out.get("anchor"),\n                       "seconds": out.get("seconds")}\n            except Exception as exc:\n                rec = {"uid": out.get("uid"), "hash_error": repr(exc)[:200]}\n            with open(os.path.join(PX_DUMP, "px_%d.jsonl" % os.getpid()), "a", encoding="utf-8") as fh:\n                fh.write(json.dumps(rec) + "\\n")\n        return out\n\n\nCI.CStudies = LocalCStudies\n\nif __name__ == "__main__":\n    CI.main()\n\'\'\'\n\nB_PROBE = r\'\'\'\nimport importlib.util, json, os, sys, time\nlauncher, anatomy, model, backbone, crop, slices = sys.argv[1:7]\nt0 = time.time()\nout = {}\ntry:\n    import setuptools\n    out["setuptools"] = setuptools.__version__\nexcept Exception as e:\n    out["setuptools"] = "MISSING " + type(e).__name__\nout["has_distutils_hack"] = importlib.util.find_spec("_distutils_hack") is not None\nspec = importlib.util.spec_from_file_location("k1_b_launcher_probe", launcher)\nL = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(L)                      # the launcher\'s own path setup, stubs and distutils guard; imports compact_infer\nimport distutils.util, torch\nout["distutils"] = "K1 shim" if os.environ.get("K1_DISTUTILS_SHIM") else str(getattr(sys.modules["distutils"], "__file__", "?"))\nimport knee_anchor_bank_v2 as KB\nseg = KB.load_model(anatomy, "cpu")              # the vendored MONAI 0.6.0 V-Net + anatomy.pt, as every B worker loads it\nimport monai\nout["monai"] = [monai.__version__, os.path.relpath(monai.__file__, L.CODE)]\nwith torch.no_grad():\n    y = seg(torch.zeros(1, 1, 128, 128))\nout["vnet_out"] = list(y.shape)\nimport timm, compact_train as CT\nout["timm"] = timm.__version__\nsd = torch.load(model, map_location="cpu", weights_only=True)\nspatial = any(k == "sp_q" or k.endswith(".sp_q") for k in sd)\nnet = CT.Net(backbone, None, freeze_blocks=16, crop=int(crop), state_spec=None, in_chans=int(slices), spatial=spatial)\nnet.load_state_dict(sd, strict=True)            # = compact_infer.load_group for this model\nout.update(spatial=spatial, params=int(sum(p.numel() for p in net.parameters())), seconds=round(time.time() - t0, 1))\nprint("K1_BPROBE " + json.dumps(out), flush=True)\n\'\'\'\n\ndef probe_b_stack(I):\n    """B\'s own stack before anything is spent on A: the launcher (stubs, distutils guard), compact_infer import, the vendored\n    MONAI V-Net with anatomy.pt (+ one forward pass), compact_train.Net with the model\'s state dict (strict).  CPU only."""\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="2", PYTHONDONTWRITEBYTECODE="1", HF_HUB_OFFLINE="1")\n    env.pop("K1_DISTUTILS_SHIM", None)\n    g = B_GEOMETRY\n    r = subprocess.run([sys.executable, "-c", B_PROBE, str(I.launcher), str(I.anatomy), str(I.models / B_MODEL["file"]),\n                        g["backbone"], str(g["crop"]), str(g["slices"])],\n                       capture_output=True, text=True, timeout=900, stdin=subprocess.DEVNULL, env=env, cwd=str(TMP))\n    line = [l for l in r.stdout.splitlines() if l.startswith("K1_BPROBE ")]\n    if r.returncode != 0 or not line:\n        raise K1Error("component-B stack probe failed rc %d: %s %s" % (r.returncode, r.stdout[-1500:], r.stderr[-2500:]))\n    return json.loads(line[0][len("K1_BPROBE "):])\n\ndef write_launcher(src_dir, code_dir, path):\n    text = LAUNCHER.replace("__SRC__", repr(str(src_dir))).replace("__CODE__", repr(str(code_dir)))\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(text)\n    return path\n\nclass Inputs:\n    pass\n\ndef prepare_inputs():\n    I = Inputs()\n    I.comp = find_mount(COMP, "test.csv")\n    I.ds = find_mount(DS_SLUG, "checkpoint-manifest.json")\n    I.helios = find_mount(HELIOS_SLUG, ANATOMY_FILE)\n    log("competition root", I.comp, "| weights", I.ds, "| heliosli", I.helios)\n    man_path = I.ds / "code_manifest.json"\n    I.manifest_sha256 = sha256_file(man_path)\n    man = json.loads(man_path.read_text())\n    man["payload_members"] = man["members"]\n    if sha256_file(I.ds / B_MODEL["file"]) != B_MODEL["sha256"]:\n        raise K1Error("model sha differs from the kernel constant")\n    if sha256_file(I.helios / ANATOMY_FILE) != ANATOMY_SHA256:\n        raise K1Error("anatomy.pt differs from the S6 manifest anchor checkpoint")\n    cfg = json.loads((I.ds / B_MODEL["run_config"]).read_text())\n    want = {"size": B_GEOMETRY["size"], "crop": B_GEOMETRY["crop"], "views": B_GEOMETRY["views"], "slices": B_GEOMETRY["slices"],\n            "phys_mm": B_GEOMETRY["phys_mm"], "backbone": B_GEOMETRY["backbone"], "spatial_attn": True, "arms": B_MODEL["arm"]}\n    bad = {k: (cfg.get(k), v) for k, v in want.items() if cfg.get(k) != v}\n    if bad:\n        raise K1Error("run_config differs from the inference arguments (trained, inferred): %s" % bad)\n    done = json.loads((I.ds / B_MODEL["done"]).read_text())\n    if done.get("model_sha256") != B_MODEL["sha256"] or os.path.basename(done.get("model", "")) != B_MODEL.get("source_file", B_MODEL["file"]):\n        raise K1Error("DONE marker does not record this model")\n    for k in ("epochs", "train_studies"):\n        if B_MODEL.get("expect_" + k) is not None and done.get(k) != B_MODEL["expect_" + k]:\n            raise K1Error("DONE marker %s=%s, expected %s" % (k, done.get(k), B_MODEL["expect_" + k]))\n    I.code_root = TMP / "k1src"\n    # code = the scored run\'s exact file set: the extras from our dataset, everything else straight from Heliosli\'s; every\n    # file is checked against the scored sha256/bytes, so the tree is byte-identical to the one that scored 0.950\n    extras = {n: man["members"][n] for n in man["from_this_dataset"]}\n    extract_payload(I.ds / EXTRAS_FILE, I.code_root, extras)\n    for n, rec in man["members"].items():\n        if n in extras:\n            continue\n        src, dst = I.helios / safe_rel(n), I.code_root / safe_rel(n)\n        if not src.is_file() or src.stat().st_size != rec["bytes"]:\n            raise K1Error("heliosli file %s missing or size differs" % n)\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(src, dst)\n        if sha256_file(dst) != rec["sha256"]:\n            raise K1Error("heliosli file %s sha differs" % n)\n    I.src = I.code_root / "heliosli"\n    I.code = I.code_root / "cloud_code"\n    for n, want_sha in SOURCE_SHA.items():\n        if sha256_file(I.src / n) != want_sha:\n            raise K1Error("heliosli/%s differs from SOURCE_SHA" % n)\n    for n, want_sha in CODE_SHA.items():\n        if sha256_file(I.code / "training" / n) != want_sha:\n            raise K1Error("cloud_code/training/%s differs from CODE_SHA" % n)\n    I.models = TMP / "k1models"\n    if I.models.exists():\n        shutil.rmtree(I.models)\n    I.models.mkdir(parents=True)\n    os.symlink(I.ds / B_MODEL["file"], I.models / B_MODEL["file"])        # the models folder holds exactly this one file\n    I.anatomy = I.helios / ANATOMY_FILE\n    I.launcher = write_launcher(I.src, I.code, TMP / "k1launch" / "k1_b_infer.py")\n    test = pd.read_csv(I.comp / "test.csv", dtype={"StudyInstanceUID": str})\n    I.test_ids = test["StudyInstanceUID"].astype(str).tolist()\n    if not I.test_ids or len(I.test_ids) != len(set(I.test_ids)):\n        raise K1Error("test.csv empty or has duplicate study ids")\n    ss = I.comp / "sample_submission.csv"\n    I.sub_cols = list(pd.read_csv(ss, nrows=1).columns) if ss.is_file() else ["StudyInstanceUID"] + LAB\n    if I.sub_cols[0] != "StudyInstanceUID" or sorted(I.sub_cols[1:]) != sorted(LAB):\n        raise K1Error("submission columns differ from the 12 findings: %s" % I.sub_cols)\n    I.cost = {}\n    for split in ("test", "train") if MODE == "parity" else ("test",):\n        sc = pd.read_csv(I.comp / (split + "_series.csv"), dtype=str, usecols=["StudyInstanceUID"])\n        I.cost.update(sc["StudyInstanceUID"].value_counts().to_dict())\n    log("inputs ok: %d test studies | MANIFEST %s | model %s %s... | %d payload files" % (\n        len(I.test_ids), I.manifest_sha256[:16], B_MODEL["file"], B_MODEL["sha256"][:16], len(man["payload_members"])))\n    _RECEIPT["inputs"] = {"comp": str(I.comp), "ds": str(I.ds), "manifest_sha256": I.manifest_sha256, "model": B_MODEL,\n                          "n_test": len(I.test_ids), "sub_cols": I.sub_cols}\n    return I\n\ndef a_fallback_cap(n):\n    return max(A_FALLBACK_MIN, math.ceil(A_FALLBACK_FRAC * n))\n\ndef b_cmd(I, split, assignment, out_dir, tmp_dir, deadline):\n    g = B_GEOMETRY\n    return [sys.executable, "-u", str(I.launcher), "--data", str(I.comp), "--split", split, "--code", str(I.code),\n            "--models", str(I.models), "--arm", B_MODEL["arm"], "--backbone", g["backbone"], "--size", str(g["size"]),\n            "--crop", str(g["crop"]), "--views", g["views"], "--slices", str(g["slices"]), "--phys-mm", str(g["phys_mm"]),\n            "--eval-stride", str(g["eval_stride"]), "--anchor-weights", str(I.anatomy), "--amp", B_AMP, "--chunk", str(g["chunk"]),\n            "--workers", str(B_WORKERS), "--item-timeout", str(g["item_timeout"]), "--assignment", str(assignment),\n            "--output", str(out_dir), "--tmp", str(tmp_dir), "--dump-logits", str(Path(out_dir) / "logits.json"),\n            "--deadline", "%.0f" % deadline]\n\ndef split_parts(ids, cost, n):\n    order = sorted(ids, key=lambda u: (-cost.get(u, 0), u))\n    parts = [order[i::n] for i in range(n)]\n    return [p[::2] + p[1::2] for p in parts if p]\n\ndef b_env(gi, px_dir=None):\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi), OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",\n               MALLOC_ARENA_MAX="2", PYTHONDONTWRITEBYTECODE="1", HF_HUB_OFFLINE="1")\n    env.pop("K1_PX_DUMP", None)\n    if px_dir is not None:\n        env["K1_PX_DUMP"] = str(px_dir)\n    return env\n\ndef collect_part(pdir, part, I):\n    rec = json.loads((pdir / "receipt.json").read_text())\n    if rec.get("status") != "PASS" or rec.get("studies") != len(part):\n        raise K1Error("%s: receipt status %s, studies %s (expected %d)" % (pdir.name, rec.get("status"), rec.get("studies"), len(part)))\n    if rec.get("amp") != B_AMP or rec.get("models") != [B_MODEL["file"]] or rec.get("spatial_models") != 1:\n        raise K1Error("%s: receipt amp %s models %s spatial %s" % (pdir.name, rec.get("amp"), rec.get("models"), rec.get("spatial_models")))\n    if rec.get("anchor_weights_sha256") != ANATOMY_SHA256:\n        raise K1Error("%s: anchor weights sha differs" % pdir.name)\n    if not isinstance(rec.get("timed_out"), bool):           # compact_infer always states it (True / False)\n        raise K1Error("%s: the receipt does not state timed_out (%r)" % (pdir.name, rec.get("timed_out")))\n    shas = rec.get("sha256")\n    if not isinstance(shas, dict) or not set(B_RECEIPT_SOURCES) <= set(shas):\n        raise K1Error("%s: the receipt\'s source shas are incomplete: %s" % (pdir.name, sorted(shas) if isinstance(shas, dict) else shas))\n    for n, s in shas.items():\n        if SOURCE_SHA.get(n) != s:\n            raise K1Error("%s: source %s sha differs in the receipt" % (pdir.name, n))\n    n_anchor = sum(int(v) for v in (rec.get("anchor_methods") or {}).values())\n    if n_anchor != rec.get("scored"):                  # every scored study must have had its SKM-TEA anchor record (review M1)\n        raise K1Error("%s: %s scored studies but %d anchor records %s" % (pdir.name, rec.get("scored"), n_anchor, rec.get("anchor_methods")))\n    logits = json.loads((pdir / "logits.json").read_text())\n    pc = pd.read_csv(pdir / "predictions.csv", dtype={"StudyInstanceUID": str})\n    if list(pc.columns) != ["StudyInstanceUID"] + LAB:\n        raise K1Error("%s: predictions.csv columns %s" % (pdir.name, list(pc.columns)))\n    pc = pc.set_index("StudyInstanceUID")\n    z = {}\n    for u in part:\n        if u not in logits:\n            continue\n        a = np.asarray(logits[u], np.float64)\n        if a.shape != (1, len(LAB)) or not np.isfinite(a).all():\n            raise K1Error("%s: logits for %s have shape %s or are non-finite" % (pdir.name, u, a.shape))\n        if u not in pc.index:\n            raise K1Error("%s: %s has logits but no predictions.csv row" % (pdir.name, u))\n        zm = a.mean(0)\n        p = 1.0 / (1.0 + np.exp(-zm))\n        d = float(np.abs(p - pc.loc[u, LAB].to_numpy(np.float64)).max())\n        if d >= 1e-5:\n            raise K1Error("%s: predictions.csv disagrees with the logits for %s (%.2e)" % (pdir.name, u, d))\n        z[u] = zm\n    extra = set(logits) - set(part)\n    if extra:\n        raise K1Error("%s: logits for studies outside the assignment: %s" % (pdir.name, sorted(extra)[:3]))\n    fallback = dict(rec.get("fallback") or {})\n    missing = [u for u in part if u not in z]\n    for u in missing:\n        fallback.setdefault(u, "no logits")\n    return z, fallback, rec\n\ndef progress_lines(logp):\n    out = []\n    for l in tail(logp, 10 ** 7).splitlines():\n        if l.startswith("{") and \'"done"\' in l:\n            try:\n                out.append(json.loads(l))\n            except Exception:\n                pass\n    return out\n\ndef b_fail_cap(n):\n    return max(B_FAIL_MIN, math.ceil(B_FAIL_FRAC * n)) if n >= RHO_MIN_N else 0\n\ndef b_done(pdir):\n    return (pdir / "receipt.json").is_file() and (pdir / "logits.json").is_file() and "WORKER_DONE" in tail(pdir / "log.txt", 3000)\n\ndef run_b(I, tag, ids, split, budget_s, px=False):\n    """B on ids over one process per GPU, one retry of failed studies.  Returns (P, Z, info, failed): failed = {uid: reason} of\n    studies still without a prediction (their P / Z rows are NaN); at most b_fail_cap(N) when tag == "B", else none.  A deadline\n    (first pass or retry) is always a HARD ERROR: a receipt whose timed_out is not False, or any fallback reason naming the\n    deadline.  px: every item\'s pixel / meta hashes are merged into <tag>/px_hashes.json.  info_sha256 (returned, so it reaches\n    k1_receipt.json) binds <tag>_info.json, which compare_parity reads for the amp and runtime checks."""\n    def deadline_named(fb):        # local, not module level: Astra\'s probe lifts run_b alone.  compact_infer\'s reason for a study\n        return [u for u, v in fb.items() if "deadline" in str(v).lower()]     # the deadline cut off: "not reached before the deadline"\n    root = OUT / tag\n    if root.exists():\n        shutil.rmtree(root)\n    root.mkdir(parents=True)\n    n_gpu = max(1, int(_RECEIPT["env"].get("n_gpu", 1)))\n    parts = split_parts(ids, I.cost, n_gpu)\n    t_start = time.time()\n    deadline = t_start + max(budget_s - B_DEADLINE_MARGIN_S, budget_s * 0.8)\n    specs, pdirs = [], []\n    for gi, part in enumerate(parts):\n        pdir = root / ("part%d" % gi)\n        pdir.mkdir()\n        (pdir / "ids.json").write_text(json.dumps(part))\n        px_dir = None\n        if px:\n            px_dir = pdir / "px"; px_dir.mkdir()\n        specs.append({"name": "%s.part%d" % (tag, gi), "cmd": b_cmd(I, split, pdir / "ids.json", pdir, TMP / ("%s_tmp%d" % (tag, gi)), deadline),\n                      "env": b_env(gi, px_dir), "cwd": str(TMP), "log": pdir / "log.txt", "done_when": (lambda d=pdir: b_done(d))})\n        pdirs.append(pdir)\n    (root / "argv_part0.json").write_text(json.dumps(specs[0]["cmd"]))\n    log("[%s] %d studies over %d GPU process(es) %s, budget %ds, compact_infer deadline +%ds" % (\n        tag, len(ids), len(parts), [len(p) for p in parts], budget_s, deadline - t_start))\n    runs = run_group(specs, budget_s, tag)\n    z, fallback, receipts = {}, {}, []\n    for pdir, part in zip(pdirs, parts):\n        zz, fb, rec = collect_part(pdir, part, I)\n        z.update(zz); fallback.update(fb)\n        receipts.append({k: rec.get(k) for k in ("studies", "scored", "seconds", "amp", "gpu", "torch", "timm", "stats", "timed_out",\n                                                 "anchor_methods", "per_study")})\n        receipts[-1]["progress"] = progress_lines(pdir / "log.txt")\n    timed = [i for i, rc in enumerate(receipts) if rc.get("timed_out") is not False]\n    if timed or deadline_named(fallback):\n        # the deadline came first: a retry cannot reach them either (review m4); no allowance ever covers a deadline.  A receipt\n        # that does not state timed_out False counts as one (collect_part already refuses it; this holds even without it)\n        raise K1Error("%s: compact_infer reached its deadline (budget %ds; timed_out not False in part(s) %s); %d studies have no "\n                      "prediction: %s" % (tag, budget_s, timed, len(fallback), dict(list(fallback.items())[:5])))\n    retry = None\n    if fallback:\n        left = t_start + budget_s - time.time()\n        log("[%s] %d fallback studies %s; %.0fs of budget left" % (tag, len(fallback), dict(list(fallback.items())[:5]), left))\n        if left < B_RETRY_MIN_S:\n            raise K1Error("%s: %d studies failed and no time is left for the retry: %s" % (tag, len(fallback), dict(list(fallback.items())[:10])))\n        rdir = root / "retry"\n        rdir.mkdir()\n        rids = sorted(fallback)\n        (rdir / "ids.json").write_text(json.dumps(rids))\n        rdeadline = time.time() + max(left - B_DEADLINE_MARGIN_S, left * 0.8)\n        if px:\n            (rdir / "px").mkdir()\n        rruns = run_group([{"name": tag + ".retry", "cmd": b_cmd(I, split, rdir / "ids.json", rdir, TMP / (tag + "_tmp_retry"), rdeadline),\n                            "env": b_env(0, (rdir / "px") if px else None), "cwd": str(TMP), "log": rdir / "log.txt",\n                            "done_when": lambda: b_done(rdir)}], left, tag)\n        zz, fb2, rec2 = collect_part(rdir, rids, I)\n        if rec2.get("timed_out") is not False or deadline_named(fb2):\n            # review C1: a retry that reaches its deadline is a HARD ERROR (PREREG section 5.1), never a capped B failure: the\n            # allowance below covers only studies that still fail after a retry that completed within its deadline\n            raise K1Error("%s: compact_infer reached its deadline in the retry (%.0fs left at its start; timed_out %r); %d of %d "\n                          "retried studies have no prediction: %s" % (tag, left, rec2.get("timed_out"), len(fb2), len(rids),\n                                                                       dict(list(fb2.items())[:5])))\n        z.update(zz)\n        retry = {"ids": rids, "first_reasons": fallback, "still_failing": fb2, "seconds": rec2.get("seconds"),\n                 "timed_out": rec2.get("timed_out"), "runs": rruns, "anchor_methods": rec2.get("anchor_methods")}\n        fallback = fb2\n    missing = [u for u in ids if u not in z]\n    failed = {u: fallback.get(u, "no logits") for u in missing}\n    cap = b_fail_cap(len(ids)) if tag == "B" else 0\n    if len(failed) > cap:\n        raise K1Error("%s: %d of %d studies have no B prediction after the retry (allowed %d): %s" % (\n            tag, len(failed), len(ids), cap, dict(list(failed.items())[:10])))\n    nan12 = np.full(len(LAB), np.nan)\n    Z = np.stack([z.get(u, nan12) for u in ids])\n    P = 1.0 / (1.0 + np.exp(-Z))\n    ok = np.array([u in z for u in ids])\n    if not np.isfinite(P[ok]).all():\n        raise K1Error("%s: non-finite B probabilities" % tag)\n    df = pd.DataFrame({"StudyInstanceUID": ids})\n    for j, l in enumerate(LAB):\n        df[l] = P[:, j]\n    for j, l in enumerate(LAB):\n        df["z|" + l] = Z[:, j]\n    df.to_csv(root / ("%s_probs.csv" % tag), index=False, float_format="%.17g")\n    info = {"seconds": round(time.time() - t_start, 1), "budget_s": round(budget_s), "parts": [len(p) for p in parts], "runs": runs,\n            "receipts": receipts, "retry": retry, "failed_after_retry": failed, "fail_cap": cap,\n            "probs_sha256": sha256_file(root / ("%s_probs.csv" % tag))}\n    if px:                                    # pixel evidence (review C3): one record per study; a retried study\'s retry record wins\n        recs, bad, clean = {}, [], lambda r: r.get("error") is None and not r.get("hash_error")\n        for f in sorted(root.glob("*/px/px_*.jsonl")):                     # part0, part1, ..., retry: sorted, so the retry is last\n            for k, line in enumerate(f.read_text(errors="replace").splitlines()):\n                try:\n                    r = json.loads(line)\n                except ValueError:\n                    r = None\n                if not isinstance(r, dict) or not isinstance(r.get("uid"), str):\n                    # a truncated line (a worker killed mid-write): diagnostic evidence never hard-fails the stage; the study is\n                    # then without a record unless another line covers it, and compare_parity FAILs missing evidence\n                    bad.append("%s:%d" % (f.relative_to(root), k + 1))\n                    continue\n                old = recs.get(r["uid"])\n                if old is None or clean(r) or not clean(old):    # a clean record is never replaced by an errored duplicate (a\n                    recs[r["uid"]] = r                           # loader restart can re-dump an item in another worker\'s file)\n        if bad:\n            log("[%s] px: %d unreadable line(s) skipped: %s" % (tag, len(bad), bad[:5]))\n        save_json(root / "px_hashes.json", recs)\n        info["px_hashes_sha256"] = sha256_file(root / "px_hashes.json")\n        info["px_studies"] = len(recs)\n        info["px_unreadable_lines"] = {"count": len(bad), "first": bad[:20]}\n    save_json(root / ("%s_info.json" % tag), info)\n    info["info_sha256"] = sha256_file(root / ("%s_info.json" % tag))   # the file holds everything above; the receipt binds it\n    log("[%s] ok: %d studies, %.0fs, retry %s, failed after retry %d (cap %d)" % (\n        tag, len(ids), info["seconds"], None if retry is None else len(retry["ids"]), len(failed), cap))\n    return P, Z, info, failed\n\ndef rank01(P):\n    """Per column: average-tie ranks 1..n divided by n, float64 (= scipy.stats.rankdata(method=\'average\') / n, gate_score.rank01)."""\n    P = np.asarray(P, np.float64)\n    n = P.shape[0]\n    out = np.empty(P.shape, np.float64)\n    for j in range(P.shape[1]):\n        v = P[:, j]\n        o = np.argsort(v, kind="mergesort")\n        s = v[o]\n        first = np.r_[True, s[1:] != s[:-1]]\n        starts = np.flatnonzero(first)\n        ends = np.r_[starts[1:], n] - 1\n        avg = (starts + ends) / 2.0 + 1.0\n        r = np.empty(n, np.float64)\n        r[o] = avg[np.cumsum(first) - 1]\n        out[:, j] = r / n\n    return out\n\ndef rank_blend(A, B, w_a=W_A):\n    return w_a * rank01(A) + (1.0 - w_a) * rank01(B)\n\ndef rho_per_finding(A, B):\n    """Spearman per finding; NaN for a constant column (exact test: np.std of equal floats can be 1e-17, not 0)."""\n    a, b = rank01(A), rank01(B)\n    return [float(np.corrcoef(a[:, j], b[:, j])[0, 1]) if np.ptp(a[:, j]) > 0 and np.ptp(b[:, j]) > 0 else float("nan")\n            for j in range(a.shape[1])]\n\ndef write_submission(ids, sub_cols, values):\n    sub = pd.DataFrame(np.asarray(values).astype(np.float32), columns=LAB)\n    sub.insert(0, "StudyInstanceUID", ids)\n    sub = sub[sub_cols]\n    if list(sub.columns) != sub_cols or sub["StudyInstanceUID"].tolist() != list(ids):\n        raise K1Error("submission column/row identity drift")\n    v = sub[LAB].to_numpy(np.float64)\n    if not np.isfinite(v).all() or (v < 0).any() or (v > 1).any():\n        raise K1Error("submission has non-finite or out-of-range values")\n    tmp = WORK / ".submission.csv.tmp"\n    sub.to_csv(tmp, index=False)\n    os.replace(tmp, WORK / "submission.csv")\n    return WORK / "submission.csv"\n\ndef blend_with_failures(A, B, failed_mask):\n    """= rank_blend(A, B) when nothing failed.  A study B failed on takes rank(A) (the 0.954 notebook\'s rule); B\'s ranks are taken\n    with those rows filled by the column mean of the scored rows (as Heliosli\'s controller fills a failed study)."""\n    failed_mask = np.asarray(failed_mask, bool)\n    if not failed_mask.any():\n        return rank_blend(A, B)\n    Bf = np.array(B, np.float64)\n    Bf[failed_mask] = np.nanmean(Bf[~failed_mask], axis=0)\n    rA = rank01(A)\n    out = W_A * rA + (1.0 - W_A) * rank01(Bf)\n    out[failed_mask] = rA[failed_mask]\n    return out\n\ndef sanity_floor(rho, n_ok):\n    """None if the floor holds (or N < RHO_MIN_N), else the reason.  A non-finite rho (a constant B column) fails it."""\n    rho = np.asarray(rho, np.float64)\n    if n_ok < RHO_MIN_N:\n        return None\n    if not np.isfinite(rho).all():\n        return "non-finite rank correlation for %s (a constant B column?)" % [LAB[j] for j in np.flatnonzero(~np.isfinite(rho))]\n    if rho.mean() < MIN_RHO_MEAN or rho.min() < MIN_RHO_FINDING:\n        return "rank corr(A, B) mean %.3f / min %.3f below %.2f / %.2f" % (rho.mean(), rho.min(), MIN_RHO_MEAN, MIN_RHO_FINDING)\n    return None\n\n\n_published_find_mount = find_mount\n\ndef mount_inventory():\n    entries = []\n    for base in (INPUT, INPUT/\'datasets\', INPUT/\'models\'):\n        if not base.is_dir():\n            continue\n        for p in sorted(base.iterdir()):\n            if p.is_dir():\n                entries.append(str(p))\n                if base != INPUT:\n                    entries.extend(str(q) for q in sorted(p.iterdir()) if q.is_dir())\n    return entries[:150]\n\ndef find_mount(slug, marker):\n    try:\n        return _published_find_mount(slug, marker)\n    except K1Error:\n        pass\n    found = []\n    blocked = {\'competitions\', \'train_series\', \'test_series\', \'study_cache\', \'__pycache__\'}\n    for current, dirs, files in os.walk(INPUT):\n        root = Path(current)\n        depth = len(root.relative_to(INPUT).parts)\n        if depth >= 6:\n            dirs[:] = []\n        else:\n            dirs[:] = [d for d in dirs if d not in blocked and d != COMP]\n        if marker in files:\n            resolved = root.resolve()\n            if resolved not in found:\n                found.append(resolved)\n    if len(found) == 1:\n        log(\'Resolved renamed/nested mount by marker:\', slug, found[0])\n        return found[0]\n    detail = {\'requested_slug\':slug, \'marker\':marker, \'candidates\':[str(p) for p in found],\n              \'attached_directories\':mount_inventory()}\n    OUT.mkdir(parents=True,exist_ok=True)\n    save_json(OUT/\'38_input_resolution.json\',detail)\n    raise K1Error(\'Missing or ambiguous input \'+slug+\'/\'+marker+\'. Candidates: \'+str(found)+\n                  \'. For anatomy.pt attach heliosli/rsna-orthofoundation-fold0-epoch7-20261004 Version 1 \'+\n                  \'(same anatomy/code dataset as successful 33). Merely attaching S6 weights is not sufficient. \'+\n                  \'See 38_input_resolution.json; do not rerun until attachments are corrected.\')\n\n'

source_path = OUT/'s6_component_b.py'
source_path.write_bytes(S6_SOURCE.encode('utf-8'))
spec = importlib.util.spec_from_file_location('rsna38_s6', source_path)
S6 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(S6)  # main and component-A runner removed at build time
S6.OUT = OUT/'worker_outputs'
S6.OUT.mkdir(exist_ok=True)
S6.TMP = Path('/tmp/rsna38_s6')
S6.TMP.mkdir(parents=True, exist_ok=True)
S6.MODE = 'parity'  # only ensures prepare_inputs reads train-series cost metadata
S6.B_GEOMETRY = dict(S6.B_GEOMETRY, chunk=96)
S6.B_WORKERS = 1  # bounded loader per GPU; model/preprocessing unchanged
S6._LOG = None
REPORT['public_source_sha256'] = '6e87350d5ae092b7a889546ca13517303523bc469f7e611ffe945913434d22fe'
REPORT['packaged_component_sha256'] = hashlib.sha256(S6_SOURCE.encode()).hexdigest()
REPORT['execution_changes'] = {'chunk': 96, 'loader_workers_per_gpu': 1,
                               'no_A_no_rank_blend_no_submission': True}
print('No training; S6 only. No Raptor or R6 checkpoint required.')


In [ ]:

try:
    REPORT['attached_directories'] = S6.mount_inventory()
    save_report()
    ds = S6.find_mount(S6.DS_SLUG, 'code_manifest.json')
    observed = S6.sha256_file(ds/'code_manifest.json')
    if observed != 'b66c574b5c0c13ccaa49d1219d41a8781431166186ef83e78cab6a9ba6b02d22':
        raise RuntimeError('S6 code manifest changed. Attach S6 version 5; do not bypass this guard: '+observed)
    I = S6.prepare_inputs()  # hashes checkpoint, anatomy and every staged code file
    train = pd.read_csv(I.comp/'train.csv', dtype={'StudyInstanceUID': str})
    gold = train.loc[train[S6.LAB].notna().all(axis=1)].set_index('StudyInstanceUID')
    if not gold.index.is_unique or set(gold.index) != set(REF['uids']) or S6.LAB != REF['labels']:
        raise RuntimeError('Gold UID/target identity mismatch')
    if not np.array_equal(gold.loc[REF['uids'], S6.LAB].to_numpy(float), np.asarray(REF['truth'])):
        raise RuntimeError('Competition gold labels differ from the embedded reference')
    if not (I.comp/'train_series').is_dir():
        raise RuntimeError('Attach competition training DICOM series, not just test files')
    REPORT['inputs'] = S6._RECEIPT['inputs']
    REPORT['environment'] = S6.probe_env()
    if not REPORT['environment'].get('cuda_available'):
        raise RuntimeError('Enable GPU T4x2 before running')
    S6._RECEIPT['env'] = REPORT['environment']
    REPORT['stack_probe'] = S6.probe_b_stack(I)  # strict state-dict + anatomy forward in disposable CPU process
    REPORT['status'] = 'PREFLIGHT_COMPLETE'
    save_report()
    print(json.dumps(REPORT, indent=2))
except BaseException as exc:
    REPORT.update(status='FAILED_PREFLIGHT', error=repr(exc), traceback=traceback.format_exc())
    save_report()
    raise


In [ ]:

try:
    # Diagnostic only. Each run enforces exact receipt/code identity, anchors and
    # complete UID coverage. A missing study is never replaced by Raptor or 0.5.
    visible_ids = I.test_ids[:3]
    p_visible, _, visible_info, failed = S6.run_b(I, 'visible', visible_ids, 'test', 1200)
    if failed:
        raise RuntimeError('S6 visible failures: '+repr(failed))
    REPORT['visible'] = visible_info
    pd.DataFrame(p_visible, index=pd.Index(visible_ids, name='StudyInstanceUID'),
                 columns=S6.LAB).to_csv(OUT/'38_s6_visible_probabilities.csv')
    save_report()
    # No hidden-test runtime extrapolation from only three studies.
    remaining = 7200 - (time.time()-T0)
    if remaining < 1800:
        raise RuntimeError('Insufficient diagnostic budget left before gold; stop safely')
    p_gold, _, gold_info, failed = S6.run_b(I, 'gold', REF['uids'], 'train', remaining)
    if failed or p_gold.shape != (58,12) or not np.isfinite(p_gold).all():
        raise RuntimeError('Incomplete/nonfinite S6 gold predictions')
    REPORT['gold'] = gold_info
    S6_GOLD = pd.DataFrame(p_gold, index=pd.Index(REF['uids'], name='StudyInstanceUID'), columns=S6.LAB)
    S6_GOLD.to_csv(OUT/'38_s6_gold_probabilities.csv')
    REPORT['status'] = 'INFERENCE_COMPLETE'
    save_report()
except BaseException as exc:
    S6.kill_all_children()
    REPORT.update(status='FAILED_INFERENCE', error=repr(exc), traceback=traceback.format_exc())
    save_report()
    raise


In [ ]:

from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

def auc_vector(y, p):
    return np.array([roc_auc_score(y[:,j], p[:,j]) if len(np.unique(y[:,j])) == 2 else np.nan
                     for j in range(y.shape[1])])

def paired_bootstrap(y, a, b, n_boot=1000):
    rng = np.random.default_rng(38019)
    deltas = []
    for _ in range(n_boot):
        ix = rng.integers(0, len(y), len(y))
        if any(len(np.unique(y[ix,j])) != 2 for j in range(y.shape[1])):
            continue  # use identical complete target sets, never variable-target macro means
        deltas.append(float(np.mean(auc_vector(y[ix],a[ix])-auc_vector(y[ix],b[ix]))))
    if not deltas:
        return {'valid_resamples': 0, 'ci95': None}
    return {'valid_resamples': len(deltas), 'ci95': np.quantile(deltas,[.025,.975]).tolist()}

def correlate(a,b):
    return float(np.corrcoef(a,b)[0,1]) if np.std(a)>0 and np.std(b)>0 else None

try:
    y = np.asarray(REF['truth'], dtype=int)
    s6p = S6_GOLD.loc[REF['uids'], REF['labels']].to_numpy(float)
    models = {name: np.asarray(REF[name],dtype=float) for name in ('r6','raptor384','proxy29B')}
    models['s6'] = s6p
    # Single preregistered diagnostic blend, NOT optimized on gold and NOT exact33.
    models['proxy29B_90_s6_10'] = .9*models['proxy29B']+.1*s6p
    for name,p in models.items():
        if p.shape != y.shape or not np.isfinite(p).all() or (p<0).any() or (p>1).any():
            raise RuntimeError('Bad prediction matrix: '+name)
    aucs = {name: auc_vector(y,p) for name,p in models.items()}
    records = []
    for j,target in enumerate(REF['labels']):
        r = {'target':target, 'positives':int(y[:,j].sum())}
        r.update({name+'_auc':float(a[j]) for name,a in aucs.items()})
        for name in ('r6','raptor384'):
            r['s6_'+name+'_spearman'] = float(spearmanr(s6p[:,j],models[name][:,j]).statistic)
            r['s6_'+name+'_signed_error_pearson'] = correlate(s6p[:,j]-y[:,j], models[name][:,j]-y[:,j])
        records.append(r)
    pd.DataFrame(records).to_csv(OUT/'38_per_target_metrics.csv', index=False)
    REPORT['macro_auc_descriptive_only'] = {n:float(np.mean(a)) for n,a in aucs.items()}
    REPORT['paired_macro_auc_delta_ci95'] = {
        's6_minus_r6': paired_bootstrap(y,s6p,models['r6']),
        's6_minus_raptor384': paired_bootstrap(y,s6p,models['raptor384']),
        'fixed_proxy_blend_minus_proxy29B': paired_bootstrap(y,models['proxy29B_90_s6_10'],models['proxy29B'])}
    cases = []
    for j,target in enumerate(REF['labels']):
        for k,u in enumerate(REF['uids']):
            cases.append({'StudyInstanceUID':u,'target':target,'gold':int(y[k,j]),
                          's6':float(s6p[k,j]),'r6':float(models['r6'][k,j]),
                          'raptor384':float(models['raptor384'][k,j]),
                          'abs_s6_r6_delta':float(abs(s6p[k,j]-models['r6'][k,j]))})
    pd.DataFrame(cases).sort_values('abs_s6_r6_delta',ascending=False).to_csv(OUT/'38_discordant_cases.csv',index=False)
    REPORT['limitations'] = [
        'Gold is small and reused upstream and in project diagnostics; confidence intervals do not remove selection bias.',
        'Signed probability-error correlation is descriptive and partly reflects calibration and shared truth.',
        'No exact notebook33 gold comparison: Raptor224 gold predictions are unavailable.',
        'Visible S6 execution is not parity against published values; no pinned reference provided.',
        'No hidden-test runtime or Private score guarantee. No blend weight selected from these metrics.']
    REPORT['status'] = 'COMPLETE'
    save_report()
    display(pd.DataFrame(records))
    print(json.dumps(REPORT['macro_auc_descriptive_only'],indent=2))
    print('COMPLETE. Download Output ZIP. Do NOT submit notebook38: it deliberately creates no submission.csv.')
except BaseException as exc:
    REPORT.update(status='FAILED_ANALYSIS',error=repr(exc),traceback=traceback.format_exc())
    save_report()
    raise
